# Coastal Erosion Analysis: Threshold Detection and Pattern Recognition

## Research Workflow Overview
1. **Annual Erosion Event Definition** - Using Net Shoreline Movement (NSM)
2. **Transect Aggregation** - Beach-scale erosion assessment
3. **Environmental Data Processing** - Feature engineering from oceanographic drivers
4. **Exploratory Data Analysis** - Driver characterization and regime identification
5. **Pattern Recognition** - Erosion driver classification
6. **Threshold Detection Models** - HMM, Random Forest, XGBoost
7. **Model Evaluation & Final Thresholds**

---
**Monsoon Year Definition**: April (Year N) to March (Year N+1)

In [1]:
import matplotlib
matplotlib.use('Agg')
# =============================================================================
# Section 1: Import Libraries and Configure Environment
# =============================================================================

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Machine Learning Libraries
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, confusion_matrix, 
                             classification_report, roc_curve)
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture  # Alternative to HMM for state detection

# XGBoost and SHAP
import xgboost as xgb
import shap

# HMM - Using Gaussian Mixture as fallback if hmmlearn unavailable
try:
    from hmmlearn import hmm
    HMM_AVAILABLE = True
except ImportError:
    HMM_AVAILABLE = False
    print("⚠ hmmlearn not available. Using Gaussian Mixture Model as alternative for state detection.")

# Plotting settings
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = [12, 6]
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14

print("✓ Libraries loaded successfully")
print(f"✓ HMM Available: {HMM_AVAILABLE}")

✓ Libraries loaded successfully
✓ HMM Available: True


## Section 2: Load Shoreline Data (NSM) and Define Erosion Labels

In [2]:
# =============================================================================
# Section 2: Load Shoreline Data and Define Erosion Labels
# =============================================================================

# Define file paths
DATA_PATH = r"D:\Kanjana\Coastal_Research_GitHub\coastalai\backend\uploads"
SHORELINE_FILE = r"D:\Kanjana\Coastal_Research_GitHub\coastalai\backend\uploads\all_stat.csv"
WAVE_FILE = r"D:\Kanjana\Coastal_Research_GitHub\coastalai\backend\uploads\2000-2025_Gobal_Ocain_waves_reanalysis.nc"
WIND_FILE = r"D:\Kanjana\Coastal_Research_GitHub\coastalai\backend\uploads\2000-2025_Global Ocean Monthly Mean Sea Surface Wind and Stress from Scatterometer and Model.nc"
CURRENT_FILE = r"D:\Kanjana\Coastal_Research_GitHub\coastalai\backend\uploads\2000-2025_Current_Data(Physics_Reanalysis).nc"

# Load shoreline statistics (transect-based)
shoreline_df = pd.read_csv(SHORELINE_FILE)

print("="*60)
print("SHORELINE DATA SUMMARY (Transect-Based)")
print("="*60)
print(f"Shape: {shoreline_df.shape}")
print(f"\nColumns: {list(shoreline_df.columns)}")
print(f"\nFirst 10 transects:")
display(shoreline_df.head(10))

# =============================================================================
# Section 2.1: Convert Transect Data to YEARLY Shoreline Changes
# Using SCE_closest_year to assign each transect's measurement to a year
# =============================================================================

# Assign transects to years based on SCE_closest_year
shoreline_df['measurement_year'] = shoreline_df['SCE_closest_year']

# Calculate ANNUAL shoreline statistics by aggregating transects per year
shoreline_annual = shoreline_df.groupby('measurement_year').agg({
    'NSM': ['mean', 'median', 'std', 'min', 'max', 'count'],
    'EPR': ['mean', 'median', 'std'],
    'LRR': ['mean', 'median', 'std'],
    'SCE': ['mean', 'max'],
}).reset_index()

# Flatten column names
shoreline_annual.columns = ['_'.join(col).strip('_') for col in shoreline_annual.columns]
shoreline_annual = shoreline_annual.rename(columns={'measurement_year': 'year'})

# Rename for consistency
shoreline_annual = shoreline_annual.rename(columns={'NSM_mean': 'annual_NSM'})

# =============================================================================
# Create Erosion Labels: 
# Since ALL transects show erosion (negative NSM), we use RELATIVE thresholds:
# - Severe Erosion: NSM below 25th percentile (more negative = more erosion)
# - Moderate Erosion: NSM between 25th-75th percentile
# - Mild Erosion/Stable: NSM above 75th percentile (less negative)
# For binary classification: Severe+Moderate = 1 (Erosion), Mild = 0 (Stable)
# =============================================================================

# Calculate thresholds based on distribution
median_nsm = shoreline_annual['annual_NSM'].median()
q25_nsm = shoreline_annual['annual_NSM'].quantile(0.25)  # More erosion (more negative)
q75_nsm = shoreline_annual['annual_NSM'].quantile(0.75)  # Less erosion

print(f"\n📊 Annual NSM Distribution:")
print(f"   Min (most erosion): {shoreline_annual['annual_NSM'].min():.2f} m")
print(f"   Q25: {q25_nsm:.2f} m")
print(f"   Median: {median_nsm:.2f} m")
print(f"   Q75: {q75_nsm:.2f} m")
print(f"   Max (least erosion): {shoreline_annual['annual_NSM'].max():.2f} m")

# Define erosion thresholds - use median as boundary
EROSION_NSM_THRESHOLD = median_nsm  # Years below median = erosion years
EROSION_PCT_THRESHOLD = 50  # Percentage-based threshold

# Create erosion status using relative threshold
shoreline_annual['Erosion_Status'] = shoreline_annual['annual_NSM'].apply(
    lambda x: 'Severe_Erosion' if x < q25_nsm 
    else ('Moderate_Erosion' if x < q75_nsm else 'Mild_Erosion')
)

# Binary label: Below median = 1 (High Erosion Year), Above median = 0 (Low Erosion Year)
shoreline_annual['Erosion_Binary'] = np.where(
    shoreline_annual['annual_NSM'] < median_nsm, 1, 0
)

print(f"\n✅ Using RELATIVE threshold: median NSM = {median_nsm:.2f} m")
print(f"   Years with NSM < median → High Erosion (1)")
print(f"   Years with NSM >= median → Low Erosion (0)")

print("\n" + "="*60)
print("YEARLY SHORELINE CHANGES (Aggregated from Transects)")
print("="*60)
print(f"\nYears with data: {len(shoreline_annual)}")
print(f"Year range: {shoreline_annual['year'].min()} - {shoreline_annual['year'].max()}")
print(f"\nAnnual Erosion Status Distribution:")
print(shoreline_annual['Erosion_Status'].value_counts())
print(f"\nYearly Shoreline Data:")
display(shoreline_annual)

# Also keep original transect-level labels
shoreline_df['Erosion_Label'] = np.where(shoreline_df['NSM'] < 0, 'Erosion', 'Accretion')
shoreline_df['Erosion_Binary'] = np.where(shoreline_df['NSM'] < 0, 1, 0)

print("\n" + "="*60)
print("TRANSECT-LEVEL EROSION DISTRIBUTION")
print("="*60)
print(f"\nErosion Label Distribution:")
print(shoreline_df['Erosion_Label'].value_counts())
print(f"\nMean NSM: {shoreline_df['NSM'].mean():.2f} m")
print(f"Median NSM: {shoreline_df['NSM'].median():.2f} m")

SHORELINE DATA SUMMARY (Transect-Based)
Shape: (109, 20)

Columns: ['id', 'SCE', 'SCE_highest_unc', 'SCE_trend', 'SCE_closest_year', 'SCE_farthest_year', 'NSM', 'NSM_highest_unc', 'NSM_trend', 'EPR', 'EPR_unc', 'EPR_trend', 'LRR', 'LR2', 'LSE', 'LCI', 'WLR', 'WR2', 'WSE', 'WCI']

First 10 transects:


,id,SCE,SCE_highest_unc,SCE_trend,SCE_closest_year,SCE_farthest_year,NSM,NSM_highest_unc,NSM_trend,EPR,EPR_unc,EPR_trend,LRR,LR2,LSE,LCI,WLR,WR2,WSE,WCI
0,1,29.96,5,accreting,2025,2011,-7.25,5,eroding,-0.48,0.47,eroding,-0.41,0.06,8.30,1.59,-0.41,0.06,1.66,1.59
1,2,26.04,5,accreting,2013,2011,-5.35,5,eroding,-0.35,0.47,stable,-0.23,0.02,7.99,1.53,-0.23,0.02,1.60,1.53
2,3,27.76,5,accreting,2025,2011,-8.32,5,eroding,-0.55,0.47,eroding,-0.50,0.12,6.70,1.29,-0.50,0.12,1.34,1.29
3,4,20.03,5,accreting,2025,2011,-6.49,5,eroding,-0.43,0.47,stable,-0.45,0.22,4.27,0.82,-0.45,0.22,0.85,0.82
4,5,19.87,5,accreting,2025,2011,-6.34,5,eroding,-0.42,0.47,stable,-0.37,0.13,4.81,0.92,-0.37,0.13,0.96,0.92
5,6,17.87,5,accreting,2025,2011,-5.73,5,eroding,-0.38,0.47,stable,-0.40,0.20,3.92,0.75,-0.40,0.20,0.78,0.75
6,7,15.52,5,accreting,2025,2011,-5.09,5,eroding,-0.34,0.47,stable,-0.27,0.10,4.04,0.78,-0.27,0.10,0.81,0.78
7,8,15.60,5,accreting,2013,2016,-5.89,5,eroding,-0.39,0.47,stable,-0.12,0.01,5.64,1.08,-0.12,0.01,1.13,1.08
8,9,18.81,5,accreting,2013,2018,-8.35,5,eroding,-0.55,0.47,eroding,-0.07,0.00,6.46,1.24,-0.07,0.00,1.29,1.24
9,10,22.30,5,accreting,2013,2022,-7.88,5,eroding,-0.52,0.47,eroding,0.00,0.00,7.09,1.36,0.00,0.00,1.42,1.36



📊 Annual NSM Distribution:
   Min (most erosion): -12.23 m
   Q25: -6.91 m
   Median: -5.66 m
   Q75: -1.49 m
   Max (least erosion): 3.54 m

✅ Using RELATIVE threshold: median NSM = -5.66 m
   Years with NSM < median → High Erosion (1)
   Years with NSM >= median → Low Erosion (0)

YEARLY SHORELINE CHANGES (Aggregated from Transects)

Years with data: 16
Year range: 2010 - 2025

Annual Erosion Status Distribution:
Erosion_Status
Moderate_Erosion    8
Severe_Erosion      4
Mild_Erosion        4
Name: count, dtype: int64

Yearly Shoreline Data:


,year,annual_NSM,NSM_median,NSM_std,NSM_min,NSM_max,NSM_count,EPR_mean,EPR_median,EPR_std,LRR_mean,LRR_median,LRR_std,SCE_mean,SCE_max,Erosion_Status,Erosion_Binary
0,2010,-6.796667,-8.280,9.413067,-15.38,3.27,3,-0.796667,-0.820,0.235867,-0.003333,0.000,0.045092,29.413333,35.59,Moderate_Erosion,1
1,2011,-9.094444,-9.540,4.690409,-16.58,2.88,27,-0.718889,-0.650,0.154928,0.115556,0.120,0.099163,30.426667,34.49,Severe_Erosion,1
2,2012,2.850909,3.080,1.113530,1.10,4.28,11,-0.187273,-0.200,0.073361,0.090000,0.100,0.029665,19.713636,20.96,Mild_Erosion,0
3,2013,-2.571429,-5.350,5.474698,-8.35,3.94,7,-0.347143,-0.350,0.151186,-0.040000,-0.010,0.110151,22.001429,28.32,Moderate_Erosion,0
4,2014,-3.170000,-3.170,10.352043,-10.49,4.15,2,-0.490000,-0.490,0.296985,0.125000,0.125,0.148492,24.775000,33.35,Moderate_Erosion,0
5,2015,2.160000,2.090,0.759430,1.24,3.27,7,-0.117143,-0.140,0.102097,0.067143,0.060,0.028702,24.985714,27.46,Mild_Erosion,0
6,2016,-5.303333,-8.640,7.855764,-10.94,3.67,3,-0.643333,-0.630,0.080829,0.083333,0.030,0.119304,28.146667,31.94,Moderate_Erosion,0
7,2017,-5.526667,-9.010,8.042216,-11.24,3.67,3,-0.713333,-0.750,0.100167,0.086667,0.050,0.100167,28.773333,33.54,Moderate_Erosion,0
8,2018,-5.790000,-9.520,7.713929,-10.93,3.08,3,-0.756667,-0.720,0.148436,0.126667,0.100,0.122202,28.430000,32.58,Moderate_Erosion,1
9,2019,-7.263333,-12.110,8.638295,-12.39,2.71,3,-0.763333,-0.800,0.081445,0.120000,0.100,0.131149,28.230000,32.71,Severe_Erosion,1



TRANSECT-LEVEL EROSION DISTRIBUTION

Erosion Label Distribution:
Erosion_Label
Erosion      69
Accretion    40
Name: count, dtype: int64

Mean NSM: -5.09 m
Median NSM: -7.88 m


In [3]:
# =============================================================================
# Section 2.2: Transect Aggregation (Beach-Scale Assessment)
# =============================================================================

# Calculate beach-scale statistics
transect_stats = {
    'Total_Transects': len(shoreline_df),
    'Mean_NSM': shoreline_df['NSM'].mean(),
    'Median_NSM': shoreline_df['NSM'].median(),
    'Std_NSM': shoreline_df['NSM'].std(),
    'Pct_Eroding': (shoreline_df['Erosion_Binary'].sum() / len(shoreline_df)) * 100,
    'Max_Erosion': shoreline_df['NSM'].min(),  # Most negative = max erosion
    'Max_Accretion': shoreline_df['NSM'].max(),
    'Mean_EPR': shoreline_df['EPR'].mean(),
    'Mean_LRR': shoreline_df['LRR'].mean()
}

print("="*60)
print("BEACH-SCALE AGGREGATION RESULTS")
print("="*60)
for key, value in transect_stats.items():
    if isinstance(value, float):
        print(f"{key}: {value:.2f}")
    else:
        print(f"{key}: {value}")

# Determine overall beach state
EROSION_THRESHOLD_PCT = 60  # % of transects that must be eroding
if transect_stats['Mean_NSM'] < 0 and transect_stats['Pct_Eroding'] > EROSION_THRESHOLD_PCT:
    beach_state = "EROSION"
else:
    beach_state = "STABLE/ACCRETION"
    
print(f"\n{'='*60}")
print(f"OVERALL BEACH STATE: {beach_state}")
print(f"{'='*60}")
print(f"Criteria: Mean NSM < 0 AND >60% transects eroding")
print(f"  - Mean NSM = {transect_stats['Mean_NSM']:.2f} m (< 0: {transect_stats['Mean_NSM'] < 0})")
print(f"  - % Eroding = {transect_stats['Pct_Eroding']:.1f}% (> 60%: {transect_stats['Pct_Eroding'] > 60})")

BEACH-SCALE AGGREGATION RESULTS
Total_Transects: 109
Mean_NSM: -5.09
Median_NSM: -7.88
Std_NSM: 6.27
Pct_Eroding: 63.30
Max_Erosion: -16.58
Max_Accretion: 4.28
Mean_EPR: -0.53
Mean_LRR: 0.03

OVERALL BEACH STATE: EROSION
Criteria: Mean NSM < 0 AND >60% transects eroding
  - Mean NSM = -5.09 m (< 0: True)
  - % Eroding = 63.3% (> 60%: True)


In [4]:
# =============================================================================
# Section 2.3: Export Yearly Shoreline Data for Frontend
# =============================================================================

# Prepare yearly shoreline data for frontend table
yearly_shoreline_export = []

for _, row in shoreline_annual.iterrows():
    yearly_shoreline_export.append({
        'year': int(row['year']),
        'annual_NSM': round(float(row['annual_NSM']), 3),
        'NSM_median': round(float(row['NSM_median']), 3),
        'NSM_std': round(float(row['NSM_std']), 3),
        'NSM_min': round(float(row['NSM_min']), 3),
        'NSM_max': round(float(row['NSM_max']), 3),
        'NSM_count': int(row['NSM_count']),
        'EPR_mean': round(float(row['EPR_mean']), 3),
        'EPR_median': round(float(row['EPR_median']), 3),
        'LRR_mean': round(float(row['LRR_mean']), 3),
        'LRR_median': round(float(row['LRR_median']), 3),
        'SCE_mean': round(float(row['SCE_mean']), 3),
        'SCE_max': round(float(row['SCE_max']), 3),
        'Erosion_Status': row['Erosion_Status'],
        'Erosion_Binary': int(row['Erosion_Binary'])
    })

print("="*60)
print("YEARLY SHORELINE DATA EXPORT")
print("="*60)
print(f"Exported {len(yearly_shoreline_export)} years of data")
print(f"\nSample (first 10 years):")
for item in yearly_shoreline_export[:10]:
    print(f"  Year {item['year']}: NSM={item['annual_NSM']:.3f}m, Status={item['Erosion_Status']}")

print(f"\n✓ Ready for frontend table display")

YEARLY SHORELINE DATA EXPORT
Exported 16 years of data

Sample (first 10 years):
  Year 2010: NSM=-6.797m, Status=Moderate_Erosion
  Year 2011: NSM=-9.094m, Status=Severe_Erosion
  Year 2012: NSM=2.851m, Status=Mild_Erosion
  Year 2013: NSM=-2.571m, Status=Moderate_Erosion
  Year 2014: NSM=-3.170m, Status=Moderate_Erosion
  Year 2015: NSM=2.160m, Status=Mild_Erosion
  Year 2016: NSM=-5.303m, Status=Moderate_Erosion
  Year 2017: NSM=-5.527m, Status=Moderate_Erosion
  Year 2018: NSM=-5.790m, Status=Moderate_Erosion
  Year 2019: NSM=-7.263m, Status=Severe_Erosion

✓ Ready for frontend table display


## Section 3: Load and Process Environmental Data (NetCDF)

In [5]:
# =============================================================================
# Section 3.1: Load Wave Data
# =============================================================================

# Load wave reanalysis data
wave_ds = xr.open_dataset(WAVE_FILE)
print("="*60)
print("WAVE DATA SUMMARY")
print("="*60)
print(f"Variables: {list(wave_ds.data_vars)}")
print(f"Dimensions: {dict(wave_ds.dims)}")
print(f"Time range: {wave_ds.time.values[0]} to {wave_ds.time.values[-1]}")
print(f"\nVariable details:")
for var in wave_ds.data_vars:
    print(f"  {var}: {wave_ds[var].dims} - {wave_ds[var].attrs.get('long_name', 'N/A')}")

WAVE DATA SUMMARY
Variables: ['VHM0', 'VTM10', 'VTM02', 'VTPK', 'VMDR', 'VPED']
Dimensions: {'time': 73041, 'latitude': 1, 'longitude': 1}
Time range: 2000-04-01T21:00:00.000000000 to 2025-03-31T21:00:00.000000000

Variable details:
  VHM0: ('time', 'latitude', 'longitude') - Spectral significant wave height (Hm0)
  VTM10: ('time', 'latitude', 'longitude') - Spectral moments (-1,0) wave period (Tm-10)
  VTM02: ('time', 'latitude', 'longitude') - Spectral moments (0,2) wave period (Tm02)
  VTPK: ('time', 'latitude', 'longitude') - Wave period at spectral peak / peak period (Tp)
  VMDR: ('time', 'latitude', 'longitude') - Mean wave direction from (Mdir)
  VPED: ('time', 'latitude', 'longitude') - Wave principal direction at spectral peak


In [6]:
# =============================================================================
# Section 3.2: Load Wind Data
# =============================================================================

# Load wind data
wind_ds = xr.open_dataset(WIND_FILE)
print("="*60)
print("WIND DATA SUMMARY")
print("="*60)
print(f"Variables: {list(wind_ds.data_vars)}")
print(f"Dimensions: {dict(wind_ds.dims)}")
print(f"Time range: {wind_ds.time.values[0]} to {wind_ds.time.values[-1]}")
print(f"\nVariable details:")
for var in wind_ds.data_vars:
    print(f"  {var}: {wind_ds[var].dims} - {wind_ds[var].attrs.get('long_name', 'N/A')}")

WIND DATA SUMMARY
Variables: ['eastward_wind', 'northward_wind', 'wind_speed', 'northward_stress', 'wind_stress_magnitude', 'eastward_stress']
Dimensions: {'time': 300, 'latitude': 1, 'longitude': 1}
Time range: 2000-04-01T00:00:00.000000000 to 2025-03-01T00:00:00.000000000

Variable details:
  eastward_wind: ('time', 'latitude', 'longitude') - Stress-equivalent wind eastward component at 10 m
  northward_wind: ('time', 'latitude', 'longitude') - Stress-equivalent wind northward component at 10 m
  wind_speed: ('time', 'latitude', 'longitude') - Stress-equivalent wind speed at 10 m
  northward_stress: ('time', 'latitude', 'longitude') - Surface wind stress northward component
  wind_stress_magnitude: ('time', 'latitude', 'longitude') - Surface wind stress magnitude
  eastward_stress: ('time', 'latitude', 'longitude') - Surface wind stress eastward component


In [7]:
# =============================================================================
# Section 3.3: Load Current Data
# =============================================================================

# Load current data
current_ds = xr.open_dataset(CURRENT_FILE)
print("="*60)
print("CURRENT DATA SUMMARY")
print("="*60)
print(f"Variables: {list(current_ds.data_vars)}")
print(f"Dimensions: {dict(current_ds.dims)}")
print(f"Time range: {current_ds.time.values[0]} to {current_ds.time.values[-1]}")
print(f"\nVariable details:")
for var in current_ds.data_vars:
    print(f"  {var}: {current_ds[var].dims} - {current_ds[var].attrs.get('long_name', 'N/A')}")

CURRENT DATA SUMMARY
Variables: ['uo', 'vo']
Dimensions: {'time': 9131, 'depth': 1, 'latitude': 1, 'longitude': 1}
Time range: 2000-04-01T00:00:00.000000000 to 2025-03-31T00:00:00.000000000

Variable details:
  uo: ('time', 'depth', 'latitude', 'longitude') - Eastward velocity
  vo: ('time', 'depth', 'latitude', 'longitude') - Northward velocity


## Section 3.4: Temporal Aggregation - Monsoon Year (April-March)

**Critical Step**: Aggregate environmental drivers to annual monsoon year resolution.

| Driver | Aggregation Strategy |
|--------|---------------------|
| Hm0 (Wave Height) | Annual maximum |
| Storm days (wave) | Count per year (Hm0 > threshold) |
| Cumulative wave energy | Annual cumulative sum |
| Wind speed | Annual max and mean |
| Wind stress | Annual mean |
| Current velocity | Annual max and cumulative |

In [8]:
# =============================================================================
# Section 3.4: Create MONTHLY Environmental Features (More Samples for Training)
# =============================================================================
# IMPORTANT: Instead of yearly aggregation (only 25 samples), we keep monthly data
# This gives ~300 samples for better model training
# Annual erosion labels will be assigned to each month within that year

def assign_monsoon_year(time):
    """
    Assign monsoon year: April (Year N) to March (Year N+1) → Monsoon Year N
    """
    month = pd.Timestamp(time).month
    year = pd.Timestamp(time).year
    if month >= 4:  # April onwards
        return year
    else:  # Jan-March belongs to previous year's monsoon
        return year - 1

# Process Wave Data - MONTHLY resolution
wave_df = wave_ds.to_dataframe().reset_index()
wave_df = wave_df.dropna(subset=['VHM0'])
wave_df['monsoon_year'] = wave_df['time'].apply(assign_monsoon_year)
wave_df['year_month'] = wave_df['time'].dt.to_period('M')

# Define storm threshold
STORM_WAVE_THRESHOLD = 2.0  # meters

# Calculate wave energy proxy
wave_df['wave_energy'] = wave_df['VHM0']**2 * wave_df['VTPK']
wave_df['is_storm_wave'] = (wave_df['VHM0'] > STORM_WAVE_THRESHOLD).astype(int)

# MONTHLY wave aggregation
wave_monthly = wave_df.groupby(['monsoon_year', 'year_month']).agg({
    'VHM0': ['max', 'mean', 'std'],
    'VTPK': ['max', 'mean'],
    'wave_energy': 'sum',
    'is_storm_wave': 'sum',
    'time': 'first'  # Keep a reference time
}).reset_index()

# Flatten column names
wave_monthly.columns = ['monsoon_year', 'year_month', 'Hm0_max', 'Hm0_mean', 'Hm0_std', 
                        'Tp_max', 'Tp_mean', 'CumWaveEnergy', 'StormDays_wave', 'time']

# Also create annual aggregation for reference
wave_annual = wave_df.groupby('monsoon_year').agg({
    'VHM0': ['max', 'mean', 'std'],
    'VTPK': ['max', 'mean'],
    'wave_energy': 'sum',
    'is_storm_wave': 'sum'
}).reset_index()
wave_annual.columns = ['monsoon_year', 'Hm0_max_annual', 'Hm0_mean_annual', 'Hm0_std_annual', 
                       'Tp_max_annual', 'Tp_mean_annual', 'CumWaveEnergy_annual', 'StormDays_wave_annual']

print("="*60)
print("WAVE DATA - MONTHLY AGGREGATION")
print("="*60)
print(f"Monthly samples: {len(wave_monthly)}")
print(f"Annual samples: {len(wave_annual)}")
print(f"\nSample rate increase: {len(wave_monthly) / len(wave_annual):.1f}x more training data!")
display(wave_monthly.head(10))

WAVE DATA - MONTHLY AGGREGATION
Monthly samples: 300
Annual samples: 25

Sample rate increase: 12.0x more training data!


,monsoon_year,year_month,Hm0_max,Hm0_mean,Hm0_std,Tp_max,Tp_mean,CumWaveEnergy,StormDays_wave,time
0,2000,2000-04,1.61,1.157940,0.235656,19.010000,12.607468,3985.690918,0,2000-04-01 21:00:00
1,2000,2000-05,2.13,1.372298,0.291076,19.020000,11.192782,5574.808594,8,2000-05-01 00:00:00
2,2000,2000-06,2.37,1.968833,0.192841,18.870001,10.738958,10089.267578,99,2000-06-01 00:00:00
3,2000,2000-07,2.39,1.815807,0.316649,18.870001,11.776733,9369.439453,77,2000-07-01 00:00:00
4,2000,2000-08,2.62,1.809758,0.364251,20.010000,11.829799,9672.284180,70,2000-08-01 00:00:00
5,2000,2000-09,2.24,1.573333,0.225913,21.370001,13.026583,7739.982910,7,2000-09-01 00:00:00
6,2000,2000-10,2.04,1.291250,0.275156,20.889999,12.228629,5027.863281,1,2000-10-01 00:00:00
7,2000,2000-11,2.31,1.027208,0.240491,19.940001,14.296499,3934.641846,4,2000-11-01 00:00:00
8,2000,2000-12,2.39,0.953589,0.273063,19.430000,12.335121,2799.014648,5,2000-12-01 00:00:00
9,2000,2001-01,1.13,0.774919,0.138728,20.500000,13.347298,2101.081543,0,2001-01-01 00:00:00


In [9]:
# =============================================================================
# Section 3.5: Wind Data - MONTHLY Aggregation
# =============================================================================

# Process Wind Data - MONTHLY resolution
wind_df = wind_ds.to_dataframe().reset_index()
wind_df = wind_df.dropna(subset=['wind_speed'])
wind_df['monsoon_year'] = wind_df['time'].apply(assign_monsoon_year)
wind_df['year_month'] = wind_df['time'].dt.to_period('M')

# Define storm wind threshold
STORM_WIND_THRESHOLD = 10.0  # m/s
wind_df['is_storm_wind'] = (wind_df['wind_speed'] > STORM_WIND_THRESHOLD).astype(int)

# MONTHLY wind aggregation
wind_monthly = wind_df.groupby(['monsoon_year', 'year_month']).agg({
    'wind_speed': ['max', 'mean', 'std'],
    'wind_stress_magnitude': ['max', 'mean'],
    'eastward_wind': 'mean',
    'northward_wind': 'mean',
    'is_storm_wind': 'sum'
}).reset_index()

# Flatten column names
wind_monthly.columns = ['monsoon_year', 'year_month', 'WindMax', 'WindMean', 'WindStd',
                        'WindStressMax', 'WindStressMean', 
                        'WindEast_mean', 'WindNorth_mean', 'StormDays_wind']

# Annual aggregation for reference
wind_annual = wind_df.groupby('monsoon_year').agg({
    'wind_speed': ['max', 'mean', 'std'],
    'wind_stress_magnitude': ['max', 'mean'],
    'is_storm_wind': 'sum'
}).reset_index()
wind_annual.columns = ['monsoon_year', 'WindMax_annual', 'WindMean_annual', 'WindStd_annual',
                       'WindStressMax_annual', 'WindStressMean_annual', 'StormDays_wind_annual']

print("="*60)
print("WIND DATA - MONTHLY AGGREGATION")
print("="*60)
print(f"Monthly samples: {len(wind_monthly)}")
display(wind_monthly.head(10))

WIND DATA - MONTHLY AGGREGATION
Monthly samples: 300


,monsoon_year,year_month,WindMax,WindMean,WindStd,WindStressMax,WindStressMean,WindEast_mean,WindNorth_mean,StormDays_wind
0,2000,2000-04,4.78,4.78,NaN,0.04,0.04,4.29,1.15,0
1,2000,2000-05,5.21,5.21,NaN,0.03,0.03,4.91,1.28,0
2,2000,2000-06,6.73,6.73,NaN,0.08,0.08,6.46,0.46,0
3,2000,2000-07,5.71,5.71,NaN,0.05,0.05,5.48,0.51,0
4,2000,2000-08,6.00,6.00,NaN,0.05,0.05,5.47,1.33,0
5,2000,2000-09,4.99,4.99,NaN,0.03,0.03,4.33,1.76,0
6,2000,2000-10,5.17,5.17,NaN,0.04,0.04,4.74,0.29,0
7,2000,2000-11,4.17,4.17,NaN,0.03,0.03,0.62,-1.57,0
8,2000,2000-12,5.64,5.64,NaN,0.07,0.07,0.74,-4.25,0
9,2000,2001-01,4.64,4.64,NaN,0.03,0.03,0.58,-3.51,0


In [10]:
# =============================================================================
# Section 3.6: Current Data - MONTHLY Aggregation
# =============================================================================

# Process Current Data - MONTHLY resolution
current_df = current_ds.to_dataframe().reset_index()
current_df = current_df.dropna(subset=['uo', 'vo'])
current_df['monsoon_year'] = current_df['time'].apply(assign_monsoon_year)
current_df['year_month'] = current_df['time'].dt.to_period('M')

# Calculate current magnitude
current_df['current_magnitude'] = np.sqrt(current_df['uo']**2 + current_df['vo']**2)

# MONTHLY current aggregation
current_monthly = current_df.groupby(['monsoon_year', 'year_month']).agg({
    'current_magnitude': ['max', 'mean', 'std', 'sum'],
    'uo': 'mean',
    'vo': 'mean'
}).reset_index()

# Flatten column names
current_monthly.columns = ['monsoon_year', 'year_month', 'UcurrMax', 'UcurrMean', 'UcurrStd', 
                           'CumCurrent', 'Ucurr_east_mean', 'Ucurr_north_mean']

# Annual aggregation for reference
current_annual = current_df.groupby('monsoon_year').agg({
    'current_magnitude': ['max', 'mean', 'std', 'sum']
}).reset_index()
current_annual.columns = ['monsoon_year', 'UcurrMax_annual', 'UcurrMean_annual', 
                          'UcurrStd_annual', 'CumCurrent_annual']

print("="*60)
print("CURRENT DATA - MONTHLY AGGREGATION")
print("="*60)
print(f"Monthly samples: {len(current_monthly)}")
display(current_monthly.head(10))

CURRENT DATA - MONTHLY AGGREGATION
Monthly samples: 300


,monsoon_year,year_month,UcurrMax,UcurrMean,UcurrStd,CumCurrent,Ucurr_east_mean,Ucurr_north_mean
0,2000,2000-04,0.321304,0.174807,0.064589,5.244216,0.100731,-0.112003
1,2000,2000-05,0.258558,0.130523,0.059600,4.046205,0.072713,-0.092481
2,2000,2000-06,0.401125,0.208043,0.077728,6.241299,0.125899,-0.162521
3,2000,2000-07,0.419346,0.177735,0.108707,5.509781,0.097482,-0.128197
4,2000,2000-08,0.359857,0.183751,0.089079,5.696272,0.068854,-0.054874
5,2000,2000-09,0.400858,0.222400,0.090340,6.672010,0.153712,-0.136642
6,2000,2000-10,0.288950,0.125059,0.070878,3.876829,0.057414,-0.049263
7,2000,2000-11,0.415590,0.216157,0.092611,6.484716,0.012920,-0.074933
8,2000,2000-12,0.261459,0.147187,0.066616,4.562797,-0.055445,-0.046546
9,2000,2001-01,0.224856,0.112684,0.040024,3.493202,-0.073441,-0.019512


In [11]:
# =============================================================================
# Section 3.7: Merge Monthly Environmental Features + Annual Context
# =============================================================================

# Merge all MONTHLY datasets
env_features_monthly = wave_monthly.merge(wind_monthly, on=['monsoon_year', 'year_month'], how='outer')
env_features_monthly = env_features_monthly.merge(current_monthly, on=['monsoon_year', 'year_month'], how='outer')

# Merge annual features (provides annual context to monthly data)
env_features_monthly = env_features_monthly.merge(wave_annual, on='monsoon_year', how='left')
env_features_monthly = env_features_monthly.merge(wind_annual, on='monsoon_year', how='left')
env_features_monthly = env_features_monthly.merge(current_annual, on='monsoon_year', how='left')

# Filter to complete years (2000-2024)
env_features_monthly = env_features_monthly[
    (env_features_monthly['monsoon_year'] >= 2000) & 
    (env_features_monthly['monsoon_year'] <= 2024)
]

# Add month indicator for seasonality
env_features_monthly['month'] = env_features_monthly['year_month'].dt.month

# Create seasonal indicators (monsoon seasonality)
def get_season(month):
    if month in [6, 7, 8, 9]:  # SW Monsoon
        return 'SW_Monsoon'
    elif month in [10, 11]:  # NE Monsoon onset
        return 'NE_Monsoon'
    elif month in [12, 1, 2]:  # NE Monsoon peak
        return 'NE_Peak'
    else:  # Pre-monsoon
        return 'Pre_Monsoon'

env_features_monthly['season'] = env_features_monthly['month'].apply(get_season)

# Create dummy variables for seasons
season_dummies = pd.get_dummies(env_features_monthly['season'], prefix='Season')
env_features_monthly = pd.concat([env_features_monthly, season_dummies], axis=1)

print("="*60)
print("MONTHLY ENVIRONMENTAL FEATURE MATRIX")
print("="*60)
print(f"Shape: {env_features_monthly.shape}")
print(f"\n✓ {len(env_features_monthly)} monthly samples (vs ~25 annual samples)")
print(f"✓ ~{len(env_features_monthly)/25:.0f}x more training data!")
print(f"\nMonsoon years covered: {env_features_monthly['monsoon_year'].min()} to {env_features_monthly['monsoon_year'].max()}")
print(f"Total months: {len(env_features_monthly)}")

# Check for missing values
missing = env_features_monthly.isnull().sum()
if missing.sum() > 0:
    print(f"\n⚠ Missing values detected:")
    print(missing[missing > 0])

print(f"\nFeatures ({len(env_features_monthly.columns)} total):")
for col in env_features_monthly.columns[:15]:
    print(f"  - {col}")
print(f"  ... and {len(env_features_monthly.columns)-15} more")

display(env_features_monthly.head(10))

MONTHLY ENVIRONMENTAL FEATURE MATRIX
Shape: (300, 47)

✓ 300 monthly samples (vs ~25 annual samples)
✓ ~12x more training data!

Monsoon years covered: 2000 to 2024
Total months: 300



⚠ Missing values detected:
WindStd                  300
WindStressMax            219
WindStressMean           219
WindStressMax_annual     216
WindStressMean_annual    216
dtype: int64

Features (47 total):
  - monsoon_year
  - year_month
  - Hm0_max
  - Hm0_mean
  - Hm0_std
  - Tp_max
  - Tp_mean
  - CumWaveEnergy
  - StormDays_wave
  - time
  - WindMax
  - WindMean
  - WindStd
  - WindStressMax
  - WindStressMean
  ... and 32 more


,monsoon_year,year_month,Hm0_max,Hm0_mean,Hm0_std,Tp_max,Tp_mean,CumWaveEnergy,StormDays_wave,time,...,UcurrMax_annual,UcurrMean_annual,UcurrStd_annual,CumCurrent_annual,month,season,Season_NE_Monsoon,Season_NE_Peak,Season_Pre_Monsoon,Season_SW_Monsoon
0,2000,2000-04,1.61,1.157940,0.235656,19.010000,12.607468,3985.690918,0,2000-04-01 21:00:00,...,0.419346,0.160645,0.084027,58.635483,4,Pre_Monsoon,False,False,True,False
1,2000,2000-05,2.13,1.372298,0.291076,19.020000,11.192782,5574.808594,8,2000-05-01 00:00:00,...,0.419346,0.160645,0.084027,58.635483,5,Pre_Monsoon,False,False,True,False
2,2000,2000-06,2.37,1.968833,0.192841,18.870001,10.738958,10089.267578,99,2000-06-01 00:00:00,...,0.419346,0.160645,0.084027,58.635483,6,SW_Monsoon,False,False,False,True
3,2000,2000-07,2.39,1.815807,0.316649,18.870001,11.776733,9369.439453,77,2000-07-01 00:00:00,...,0.419346,0.160645,0.084027,58.635483,7,SW_Monsoon,False,False,False,True
4,2000,2000-08,2.62,1.809758,0.364251,20.010000,11.829799,9672.284180,70,2000-08-01 00:00:00,...,0.419346,0.160645,0.084027,58.635483,8,SW_Monsoon,False,False,False,True
5,2000,2000-09,2.24,1.573333,0.225913,21.370001,13.026583,7739.982910,7,2000-09-01 00:00:00,...,0.419346,0.160645,0.084027,58.635483,9,SW_Monsoon,False,False,False,True
6,2000,2000-10,2.04,1.291250,0.275156,20.889999,12.228629,5027.863281,1,2000-10-01 00:00:00,...,0.419346,0.160645,0.084027,58.635483,10,NE_Monsoon,True,False,False,False
7,2000,2000-11,2.31,1.027208,0.240491,19.940001,14.296499,3934.641846,4,2000-11-01 00:00:00,...,0.419346,0.160645,0.084027,58.635483,11,NE_Monsoon,True,False,False,False
8,2000,2000-12,2.39,0.953589,0.273063,19.430000,12.335121,2799.014648,5,2000-12-01 00:00:00,...,0.419346,0.160645,0.084027,58.635483,12,NE_Peak,False,True,False,False
9,2000,2001-01,1.13,0.774919,0.138728,20.500000,13.347298,2101.081543,0,2001-01-01 00:00:00,...,0.419346,0.160645,0.084027,58.635483,1,NE_Peak,False,True,False,False


In [12]:
# =============================================================================
# Section 3.8: Handle Missing Values and Merge with ACTUAL Shoreline Data
# =============================================================================

# Fill missing values using interpolation
numeric_cols = env_features_monthly.select_dtypes(include=[np.number]).columns
env_features_monthly[numeric_cols] = env_features_monthly[numeric_cols].interpolate(method='linear')
env_features_monthly = env_features_monthly.ffill().bfill()

# =============================================================================
# IMPORTANT: Use ACTUAL YEARLY SHORELINE DATA (from SCE_closest_year)
# Environmental data = MONTHLY (env_features_monthly)
# Shoreline data = YEARLY (shoreline_annual)
# =============================================================================

# Calculate normalized intensity scores (for monthly data)
env_features_monthly['wave_intensity'] = (
    (env_features_monthly['Hm0_max'] - env_features_monthly['Hm0_max'].mean()) / 
    env_features_monthly['Hm0_max'].std()
)
env_features_monthly['wind_intensity'] = (
    (env_features_monthly['WindMax'] - env_features_monthly['WindMax'].mean()) / 
    env_features_monthly['WindMax'].std()
)
env_features_monthly['current_intensity'] = (
    (env_features_monthly['UcurrMax'] - env_features_monthly['UcurrMax'].mean()) / 
    env_features_monthly['UcurrMax'].std()
)

# Monthly environmental forcing index
env_features_monthly['env_forcing_index'] = (
    env_features_monthly['wave_intensity'] + 
    env_features_monthly['wind_intensity'] + 
    env_features_monthly['current_intensity']
) / 3

# =============================================================================
# Merge ACTUAL yearly shoreline erosion labels with monthly environmental data
# Each year's REAL erosion status is assigned to all 12 months in that year
# =============================================================================

# Prepare shoreline_annual for merging (rename year to monsoon_year)
shoreline_annual_merge = shoreline_annual.copy()
shoreline_annual_merge = shoreline_annual_merge.rename(columns={'year': 'monsoon_year'})

# Merge actual shoreline data with monthly environmental data
env_features_monthly = env_features_monthly.merge(
    shoreline_annual_merge[['monsoon_year', 'annual_NSM', 'Erosion_Binary', 'Erosion_Status', 'NSM_count']], 
    on='monsoon_year', 
    how='left',
    suffixes=('_env', '_shore')
)

# Use actual shoreline erosion labels
if 'Erosion_Binary_shore' in env_features_monthly.columns:
    env_features_monthly['Erosion_Label'] = env_features_monthly['Erosion_Binary_shore']
    env_features_monthly = env_features_monthly.drop(columns=['Erosion_Binary_shore'], errors='ignore')
elif 'Erosion_Binary' in env_features_monthly.columns:
    env_features_monthly['Erosion_Label'] = env_features_monthly['Erosion_Binary']

# Handle years without shoreline data (fill with 0 - stable)
env_features_monthly['Erosion_Label'] = env_features_monthly['Erosion_Label'].fillna(0).astype(int)
env_features_monthly['annual_NSM'] = env_features_monthly['annual_NSM'].fillna(0)

print("="*60)
print("DATA FREQUENCY SUMMARY")
print("="*60)
print(f"\n📊 ENVIRONMENTAL DATA: MONTHLY Frequency")
print(f"   - {len(env_features_monthly)} monthly samples")
print(f"   - Features: Wave (Hm0), Wind, Current per month")

print(f"\n📊 SHORELINE DATA: YEARLY Frequency")
print(f"   - {len(shoreline_annual)} yearly samples")
print(f"   - Features: NSM, EPR, LRR aggregated from transects")

print(f"\n🔗 MERGED DATA: Monthly env + Yearly erosion labels")
print(f"   - Each month gets its year's erosion status")
print(f"   - Years with shoreline data: {env_features_monthly['monsoon_year'].nunique()}")

print(f"\n✓ ACTUAL Shoreline Erosion Labels (from SCE_closest_year):")
print(f"   Erosion months: {env_features_monthly['Erosion_Label'].sum()} ({env_features_monthly['Erosion_Label'].mean()*100:.1f}%)")
print(f"   Stable/Accretion months: {len(env_features_monthly) - env_features_monthly['Erosion_Label'].sum()}")

# Show which years have actual shoreline data
years_with_shore_data = env_features_monthly[env_features_monthly['annual_NSM'] != 0]['monsoon_year'].unique()
print(f"\n📅 Years with ACTUAL shoreline measurements: {sorted(years_with_shore_data)}")

# =============================================================================
# Create annual aggregated environmental data (env_features)
# =============================================================================

# Calculate annual statistics for each year
annual_stats = env_features_monthly.groupby('monsoon_year').agg({
    'Hm0_max': 'max',   # Annual max wave height
    'WindMax': 'max',   # Annual max wind
    'UcurrMax': 'max',  # Annual max current
    'env_forcing_index': 'max',  # Max monthly forcing that year
    'Erosion_Label': 'first',  # Use actual yearly erosion label
    'annual_NSM': 'first'  # Actual NSM for the year
}).reset_index()

annual_stats.columns = ['monsoon_year', 'Hm0_max_year', 'WindMax_year', 
                        'UcurrMax_year', 'max_forcing', 'Erosion_Label', 'annual_NSM']

# Create annual normalized scores
annual_stats['annual_wave_norm'] = (
    (annual_stats['Hm0_max_year'] - annual_stats['Hm0_max_year'].mean()) / 
    annual_stats['Hm0_max_year'].std()
)
annual_stats['annual_wind_norm'] = (
    (annual_stats['WindMax_year'] - annual_stats['WindMax_year'].mean()) / 
    annual_stats['WindMax_year'].std()
)
annual_stats['annual_curr_norm'] = (
    (annual_stats['UcurrMax_year'] - annual_stats['UcurrMax_year'].mean()) / 
    annual_stats['UcurrMax_year'].std()
)

annual_stats['annual_forcing_composite'] = (
    annual_stats['annual_wave_norm'] + 
    annual_stats['annual_wind_norm'] + 
    annual_stats['annual_curr_norm']
) / 3

# Create env_features (annual) for backward compatibility
env_features = annual_stats.copy()

# Add more annual features from the original annual aggregations
env_features = env_features.merge(wave_annual, on='monsoon_year', how='left')
env_features = env_features.merge(wind_annual, on='monsoon_year', how='left')
env_features = env_features.merge(current_annual, on='monsoon_year', how='left')

# Create backward compatible column names
env_features['Hm0_max'] = env_features['Hm0_max_year']
env_features['WindMax'] = env_features['WindMax_year']
env_features['UcurrMax'] = env_features['UcurrMax_year']

# Fill available columns
for col in ['Hm0_mean', 'WindMean', 'UcurrMean', 'CumCurrent', 'WindStressMean', 
            'StormDays_wave', 'CumWaveEnergy']:
    if col + '_annual' in env_features.columns:
        env_features[col] = env_features[col + '_annual']
    elif col not in env_features.columns:
        env_features[col] = 0

# Fill NaN
env_features = env_features.fillna(0)

# Add intensity scores
env_features['wave_intensity'] = env_features['annual_wave_norm']
env_features['wind_intensity'] = env_features['annual_wind_norm'] 
env_features['current_intensity'] = env_features['annual_curr_norm']
env_features['env_forcing_index'] = env_features['annual_forcing_composite']

print(f"\n" + "="*60)
print("FINAL DATA STRUCTURE SUMMARY")
print("="*60)
print(f"\n📊 env_features_monthly (MONTHLY Environmental + Yearly Erosion Labels):")
print(f"   - Rows: {len(env_features_monthly)} monthly samples")
print(f"   - Each row: environmental data for one month")
print(f"   - Erosion_Label: ACTUAL yearly erosion from shoreline_annual")
print(f"   - Columns: {list(env_features_monthly.columns)[:10]}...")

print(f"\n📊 env_features (ANNUAL Environmental + Yearly Erosion Labels):")
print(f"   - Rows: {len(env_features)} yearly samples")
print(f"   - Columns: {list(env_features.columns)[:10]}...")

print(f"\n📊 shoreline_annual (YEARLY Shoreline from SCE_closest_year):")
print(f"   - Rows: {len(shoreline_annual)} years")
print(f"   - Features: annual_NSM, EPR_mean, LRR_mean, Erosion_Binary")

display(env_features_monthly[['monsoon_year', 'year_month', 'Hm0_max', 'WindMax', 
                              'UcurrMax', 'annual_NSM', 'Erosion_Label']].head(15))

DATA FREQUENCY SUMMARY

📊 ENVIRONMENTAL DATA: MONTHLY Frequency
   - 300 monthly samples
   - Features: Wave (Hm0), Wind, Current per month

📊 SHORELINE DATA: YEARLY Frequency
   - 16 yearly samples
   - Features: NSM, EPR, LRR aggregated from transects

🔗 MERGED DATA: Monthly env + Yearly erosion labels
   - Each month gets its year's erosion status
   - Years with shoreline data: 25

✓ ACTUAL Shoreline Erosion Labels (from SCE_closest_year):
   Erosion months: 84 (28.0%)
   Stable/Accretion months: 216

📅 Years with ACTUAL shoreline measurements: [np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

FINAL DATA STRUCTURE SUMMARY

📊 env_features_monthly (MONTHLY Environmental + Yearly Erosion Labels):
   - Rows: 300 monthly samples
   - Each row: environmental data for one month
   - Erosion_Label: AC

,monsoon_year,year_month,Hm0_max,WindMax,UcurrMax,annual_NSM,Erosion_Label
0,2000,2000-04,1.61,4.78,0.321304,0.0,0
1,2000,2000-05,2.13,5.21,0.258558,0.0,0
2,2000,2000-06,2.37,6.73,0.401125,0.0,0
3,2000,2000-07,2.39,5.71,0.419346,0.0,0
4,2000,2000-08,2.62,6.00,0.359857,0.0,0
5,2000,2000-09,2.24,4.99,0.400858,0.0,0
6,2000,2000-10,2.04,5.17,0.288950,0.0,0
7,2000,2000-11,2.31,4.17,0.415590,0.0,0
8,2000,2000-12,2.39,5.64,0.261459,0.0,0
9,2000,2001-01,1.13,4.64,0.224856,0.0,0


## Section 4: Exploratory Data Analysis (EDA)

Analyze the relationship between environmental drivers and erosion events:
- Boxplots comparing erosion vs stable years
- Correlation matrices
- PCA for regime identification
- Clustering analysis

In [13]:
# =============================================================================
# Section 4.1: Boxplots - Erosion vs Stable Years
# =============================================================================

# Key drivers to compare (using columns available in annual env_features)
# Check available columns and use only existing ones
available_cols = env_features.columns.tolist()
print("Available columns in env_features:", available_cols)

# Define key drivers that exist in the data
key_drivers = []
potential_drivers = ['Hm0_max', 'CumWaveEnergy', 'StormDays_wave', 
                     'WindMax', 'WindStressMean', 'UcurrMax', 'CumCurrent', 
                     'Hm0_mean', 'WindMean', 'UcurrMean']

for driver in potential_drivers:
    if driver in available_cols:
        key_drivers.append(driver)
    elif driver + '_annual' in available_cols:
        key_drivers.append(driver + '_annual')

print(f"\nUsing drivers: {key_drivers}")

# Limit to 8 drivers for the subplot layout
key_drivers = key_drivers[:8]

fig, axes = plt.subplots(2, 4, figsize=(16, 10))
axes = axes.flatten()

for i, driver in enumerate(key_drivers):
    ax = axes[i]
    erosion_data = env_features[env_features['Erosion_Label'] == 1][driver]
    stable_data = env_features[env_features['Erosion_Label'] == 0][driver]
    
    bp = ax.boxplot([stable_data, erosion_data], 
                    labels=['Stable', 'Erosion'],
                    patch_artist=True)
    bp['boxes'][0].set_facecolor('lightgreen')
    bp['boxes'][1].set_facecolor('salmon')
    
    ax.set_title(driver.replace('_annual', ''))
    ax.set_ylabel('Value')

# Hide unused axes if less than 8 drivers
for j in range(len(key_drivers), 8):
    axes[j].set_visible(False)
    
plt.suptitle('Environmental Drivers: Erosion vs Stable Years', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Statistical summary
print("="*60)
print("STATISTICAL COMPARISON: Erosion vs Stable Years")
print("="*60)
for driver in key_drivers:
    erosion_mean = env_features[env_features['Erosion_Label'] == 1][driver].mean()
    stable_mean = env_features[env_features['Erosion_Label'] == 0][driver].mean()
    diff_pct = ((erosion_mean - stable_mean) / stable_mean) * 100 if stable_mean != 0 else 0
    print(f"{driver.replace('_annual', ''):25s}: Erosion={erosion_mean:.3f}, Stable={stable_mean:.3f}, Diff={diff_pct:+.1f}%")

Available columns in env_features: ['monsoon_year', 'Hm0_max_year', 'WindMax_year', 'UcurrMax_year', 'max_forcing', 'Erosion_Label', 'annual_NSM', 'annual_wave_norm', 'annual_wind_norm', 'annual_curr_norm', 'annual_forcing_composite', 'Hm0_max_annual', 'Hm0_mean_annual', 'Hm0_std_annual', 'Tp_max_annual', 'Tp_mean_annual', 'CumWaveEnergy_annual', 'StormDays_wave_annual', 'WindMax_annual', 'WindMean_annual', 'WindStd_annual', 'WindStressMax_annual', 'WindStressMean_annual', 'StormDays_wind_annual', 'UcurrMax_annual', 'UcurrMean_annual', 'UcurrStd_annual', 'CumCurrent_annual', 'Hm0_max', 'WindMax', 'UcurrMax', 'Hm0_mean', 'WindMean', 'UcurrMean', 'CumCurrent', 'WindStressMean', 'StormDays_wave', 'CumWaveEnergy', 'wave_intensity', 'wind_intensity', 'current_intensity', 'env_forcing_index']

Using drivers: ['Hm0_max', 'CumWaveEnergy', 'StormDays_wave', 'WindMax', 'WindStressMean', 'UcurrMax', 'CumCurrent', 'Hm0_mean', 'WindMean', 'UcurrMean']


STATISTICAL COMPARISON: Erosion vs Stable Years
Hm0_max                  : Erosion=3.024, Stable=2.748, Diff=+10.1%
CumWaveEnergy            : Erosion=59714.094, Stable=57980.172, Diff=+3.0%
StormDays_wave           : Erosion=185.286, Stable=153.389, Diff=+20.8%
WindMax                  : Erosion=6.596, Stable=6.477, Diff=+1.8%
WindStressMean           : Erosion=0.000, Stable=0.015, Diff=-100.0%
UcurrMax                 : Erosion=0.484, Stable=0.463, Diff=+4.4%
CumCurrent               : Erosion=53.995, Stable=54.133, Diff=-0.3%
Hm0_mean                 : Erosion=1.199, Stable=1.186, Diff=+1.1%


In [14]:
# =============================================================================
# Section 4.1.1: Export Boxplot Data for Frontend
# =============================================================================

# Prepare boxplot comparison data for frontend
boxplot_export = []

for driver in key_drivers:
    erosion_data = env_features[env_features['Erosion_Label'] == 1][driver]
    stable_data = env_features[env_features['Erosion_Label'] == 0][driver]
    
    # Calculate statistics
    erosion_mean = erosion_data.mean()
    stable_mean = stable_data.mean()
    diff_pct = ((erosion_mean - stable_mean) / stable_mean) * 100 if stable_mean != 0 else 0
    
    # Calculate boxplot statistics for both groups
    stable_stats = {
        'min': float(stable_data.min()),
        'q1': float(stable_data.quantile(0.25)),
        'median': float(stable_data.median()),
        'q3': float(stable_data.quantile(0.75)),
        'max': float(stable_data.max()),
        'mean': float(stable_mean)
    }
    
    erosion_stats = {
        'min': float(erosion_data.min()),
        'q1': float(erosion_data.quantile(0.25)),
        'median': float(erosion_data.median()),
        'q3': float(erosion_data.quantile(0.75)),
        'max': float(erosion_data.max()),
        'mean': float(erosion_mean)
    }
    
    boxplot_export.append({
        'name': driver.replace('_annual', ''),
        'stableMean': float(stable_mean),
        'erosionMean': float(erosion_mean),
        'diffPct': float(diff_pct),
        'comparison': [
            {
                'category': 'Stable',
                'min': stable_stats['min'],
                'q1': stable_stats['q1'],
                'median': stable_stats['median'],
                'q3': stable_stats['q3'],
                'max': stable_stats['max'],
                'mean': stable_stats['mean'],
                'whiskerRange': stable_stats['max'] - stable_stats['min'],
                'iqrRange': stable_stats['q3'] - stable_stats['q1']
            },
            {
                'category': 'Erosion',
                'min': erosion_stats['min'],
                'q1': erosion_stats['q1'],
                'median': erosion_stats['median'],
                'q3': erosion_stats['q3'],
                'max': erosion_stats['max'],
                'mean': erosion_stats['mean'],
                'whiskerRange': erosion_stats['max'] - erosion_stats['min'],
                'iqrRange': erosion_stats['q3'] - erosion_stats['q1']
            }
        ]
    })

print("="*60)
print("BOXPLOT DATA EXPORT")
print("="*60)
print(f"Exported {len(boxplot_export)} driver comparisons")
for item in boxplot_export[:3]:
    print(f"\n{item['name']}:")
    print(f"  Stable Mean: {item['stableMean']:.3f}, Erosion Mean: {item['erosionMean']:.3f}")
    print(f"  Difference: {item['diffPct']:+.1f}%")

BOXPLOT DATA EXPORT
Exported 8 driver comparisons

Hm0_max:
  Stable Mean: 2.748, Erosion Mean: 3.024
  Difference: +10.1%

CumWaveEnergy:
  Stable Mean: 57980.172, Erosion Mean: 59714.094
  Difference: +3.0%

StormDays_wave:
  Stable Mean: 153.389, Erosion Mean: 185.286
  Difference: +20.8%


In [15]:
# =============================================================================
# Section 4.2: Correlation Matrix
# =============================================================================

# Select features for correlation analysis
corr_features = ['Hm0_max', 'Hm0_mean', 'CumWaveEnergy', 'StormDays_wave',
                 'WindMax', 'WindMean', 'WindStressMean',
                 'UcurrMax', 'UcurrMean', 'CumCurrent', 'Erosion_Label']

corr_matrix = env_features[corr_features].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', 
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5, ax=ax)
plt.title('Correlation Matrix: Environmental Drivers and Erosion', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Identify strongest correlations with erosion
erosion_corr = corr_matrix['Erosion_Label'].drop('Erosion_Label').sort_values(key=abs, ascending=False)
print("="*60)
print("CORRELATION WITH EROSION (sorted by strength)")
print("="*60)
for feature, corr in erosion_corr.items():
    print(f"{feature:20s}: {corr:+.3f}")

CORRELATION WITH EROSION (sorted by strength)
Hm0_max             : +0.442
WindStressMean      : -0.387
StormDays_wave      : +0.208
WindMean            : -0.153
CumWaveEnergy       : +0.147
WindMax             : +0.140
Hm0_mean            : +0.112
UcurrMax            : +0.104
UcurrMean           : -0.017
CumCurrent          : -0.016


In [16]:
# =============================================================================
# Section 4.3: PCA Analysis for Regime Identification
# =============================================================================

# Prepare features for PCA
pca_features = ['Hm0_max', 'Hm0_mean', 'CumWaveEnergy', 'StormDays_wave',
                'WindMax', 'WindMean', 'UcurrMax', 'UcurrMean', 'CumCurrent']

X_pca = env_features[pca_features].values
X_scaled = StandardScaler().fit_transform(X_pca)

# Apply PCA
pca = PCA(n_components=3)
pca_result = pca.fit_transform(X_scaled)

# Add PCA results to dataframe
env_features['PC1'] = pca_result[:, 0]
env_features['PC2'] = pca_result[:, 1]
env_features['PC3'] = pca_result[:, 2]

# Plot PCA results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PC1 vs PC2 colored by erosion label
scatter = axes[0].scatter(env_features['PC1'], env_features['PC2'], 
                          c=env_features['Erosion_Label'], cmap='RdYlGn_r', 
                          s=100, alpha=0.7, edgecolors='black')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
axes[0].set_title('PCA: Erosion vs Stable Years')
axes[0].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[0].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
cbar = plt.colorbar(scatter, ax=axes[0])
cbar.set_label('Erosion (1) / Stable (0)')

# Year annotations
for i, row in env_features.iterrows():
    axes[0].annotate(str(int(row['monsoon_year'])), 
                     (row['PC1'], row['PC2']), fontsize=8, alpha=0.7)

# Explained variance
axes[1].bar(range(1, 4), pca.explained_variance_ratio_ * 100, 
            color=['steelblue', 'coral', 'green'], alpha=0.7)
axes[1].set_xlabel('Principal Component')
axes[1].set_ylabel('Explained Variance (%)')
axes[1].set_title('PCA Explained Variance')
axes[1].set_xticks([1, 2, 3])

plt.tight_layout()
plt.show()

# PCA loadings interpretation
print("="*60)
print("PCA LOADINGS (Feature Contribution to Each PC)")
print("="*60)
loadings_df = pd.DataFrame(pca.components_.T, 
                           columns=['PC1', 'PC2', 'PC3'], 
                           index=pca_features)
display(loadings_df.round(3))

print(f"\nTotal variance explained by PC1-PC3: {pca.explained_variance_ratio_.sum()*100:.1f}%")

PCA LOADINGS (Feature Contribution to Each PC)


,PC1,PC2,PC3
Hm0_max,0.201,0.504,0.527
Hm0_mean,0.418,-0.253,0.143
CumWaveEnergy,0.317,-0.318,0.460
StormDays_wave,0.361,-0.330,0.270
WindMax,0.267,0.130,-0.126
WindMean,0.316,-0.290,-0.322
UcurrMax,0.268,0.563,0.095
UcurrMean,0.395,0.163,-0.379
CumCurrent,0.394,0.166,-0.381



Total variance explained by PC1-PC3: 82.3%


## Section 5: Pattern Recognition - Erosion Driver Classification

Categorize erosion events by dominant driver behavior:
- Wave-dominated erosion
- Wind-dominated erosion
- Current-dominated erosion
- Combined driver erosion

In [17]:
# =============================================================================
# Section 5.1: Driver Dominance Classification using K-Means Clustering
# =============================================================================

# Use standardized intensity scores for clustering
cluster_features = ['wave_intensity', 'wind_intensity', 'current_intensity']
X_cluster = env_features[cluster_features].values

# Apply K-Means clustering (5 clusters for different forcing regimes)
n_clusters = 5
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
env_features['Forcing_Cluster'] = kmeans.fit_predict(X_cluster)

# Define cluster labels based on centroid characteristics
cluster_centers = pd.DataFrame(kmeans.cluster_centers_, 
                               columns=['Wave', 'Wind', 'Current'])

print("="*60)
print("FORCING REGIME CLUSTERS")
print("="*60)
print("\nCluster Centroids (Standardized Intensity):")
display(cluster_centers.round(3))

# Assign descriptive labels based on dominant forcing
def classify_regime(row):
    wave = row['wave_intensity']
    wind = row['wind_intensity'] 
    current = row['current_intensity']
    
    threshold = 0.5  # Threshold for "high" intensity
    
    high_wave = wave > threshold
    high_wind = wind > threshold
    high_current = current > threshold
    
    if high_wave and high_wind and high_current:
        return 'Wave-Wind-Current Combined'
    elif high_wave and high_wind:
        return 'Wave-Wind Combined'
    elif high_wave and high_current:
        return 'Wave-Current Combined'
    elif high_wind and high_current:
        return 'Wind-Current Combined'
    elif high_wave:
        return 'Wave-Dominated'
    elif high_wind:
        return 'Wind-Dominated'
    elif high_current:
        return 'Current-Dominated'
    else:
        return 'Low-Energy'

env_features['Forcing_Regime'] = env_features.apply(classify_regime, axis=1)

# Display regime distribution
print("\nForcing Regime Distribution:")
print(env_features['Forcing_Regime'].value_counts())

# Plot clustering results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 3D-style scatter using PC1 and PC2
colors = env_features['Forcing_Cluster']
scatter = axes[0].scatter(env_features['wave_intensity'], 
                          env_features['current_intensity'],
                          c=colors, cmap='viridis', s=100, alpha=0.7)
axes[0].set_xlabel('Wave Intensity (Standardized)')
axes[0].set_ylabel('Current Intensity (Standardized)')
axes[0].set_title('Forcing Regime Clusters')
axes[0].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[0].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
plt.colorbar(scatter, ax=axes[0], label='Cluster')

# Regime distribution by erosion
regime_erosion = env_features.groupby(['Forcing_Regime', 'Erosion_Label']).size().unstack(fill_value=0)
regime_erosion.plot(kind='bar', ax=axes[1], color=['lightgreen', 'salmon'])
axes[1].set_xlabel('Forcing Regime')
axes[1].set_ylabel('Count')
axes[1].set_title('Erosion by Forcing Regime')
axes[1].legend(['Stable', 'Erosion'])
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

FORCING REGIME CLUSTERS

Cluster Centroids (Standardized Intensity):


,Wave,Wind,Current
0,0.610,-0.266,-0.462
1,-0.817,-1.052,-0.479
2,1.701,0.103,1.660
3,-0.522,0.780,-0.568
4,0.517,1.116,0.853



Forcing Regime Distribution:
Forcing_Regime
Low-Energy                    10
Wind-Dominated                 5
Wave-Dominated                 4
Wave-Wind-Current Combined     4
Wind-Current Combined          1
Wave-Current Combined          1
Name: count, dtype: int64


In [18]:
# =============================================================================
# Section 5.2: Export Forcing Regime Data for Frontend
# =============================================================================

# Calculate regime distribution with erosion breakdown
regime_counts = env_features['Forcing_Regime'].value_counts()
regime_erosion_counts = env_features.groupby(['Forcing_Regime', 'Erosion_Label']).size().unstack(fill_value=0)

# Prepare forcing regime data for export
forcing_regimes_export = []
for regime in regime_counts.index:
    total_count = regime_counts[regime]
    erosion_count = regime_erosion_counts.loc[regime, 1] if 1 in regime_erosion_counts.columns and regime in regime_erosion_counts.index else 0
    erosion_rate = (erosion_count / total_count * 100) if total_count > 0 else 0
    
    forcing_regimes_export.append({
        'regime': regime,
        'count': int(total_count),
        'percentage': round(total_count / len(env_features) * 100, 1),
        'erosionRate': round(erosion_rate, 1),
        'color': {
            'Low-Energy': '#22c55e',
            'Wind-Dominated': '#3b82f6',
            'Wave-Dominated': '#f97316',
            'Wave-Wind-Current Combined': '#ef4444',
            'Wave-Current Combined': '#8b5cf6',
            'Wind-Current Combined': '#06b6d4',
            'Current-Dominated': '#ec4899'
        }.get(regime, '#94a3b8')
    })

print("="*60)
print("FORCING REGIME EXPORT DATA")
print("="*60)
for regime_data in forcing_regimes_export:
    print(f"{regime_data['regime']:30s}: {regime_data['count']:2d} years ({regime_data['percentage']:4.1f}%), Erosion Rate: {regime_data['erosionRate']:4.1f}%")

print(f"\n✓ Prepared {len(forcing_regimes_export)} forcing regime categories for export")

FORCING REGIME EXPORT DATA
Low-Energy                    : 10 years (40.0%), Erosion Rate: 10.0%
Wind-Dominated                :  5 years (20.0%), Erosion Rate: 20.0%
Wave-Dominated                :  4 years (16.0%), Erosion Rate: 50.0%
Wave-Wind-Current Combined    :  4 years (16.0%), Erosion Rate: 50.0%
Wind-Current Combined         :  1 years ( 4.0%), Erosion Rate: 100.0%
Wave-Current Combined         :  1 years ( 4.0%), Erosion Rate:  0.0%

✓ Prepared 6 forcing regime categories for export


## Section 6: Threshold Detection Models

Implementing three models as per methodology:
1. **Hidden Markov Model (HMM)** - Sequential state detection with transition probabilities
2. **Random Forest (RF)** - Feature importance and threshold extraction
3. **XGBoost (XGB)** - High-accuracy classification with SHAP analysis

In [19]:
# =============================================================================
# Section 6.1: Prepare Feature Matrix for Modeling
# Using MONTHLY data for more training samples
# =============================================================================

from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.model_selection import LeaveOneGroupOut, RepeatedStratifiedKFold

# Select key features for modeling (monthly features + seasonal + annual context)
monthly_features = ['Hm0_max', 'Hm0_mean', 'CumWaveEnergy', 'StormDays_wave',
                    'WindMax', 'WindMean', 'WindStressMean',
                    'UcurrMax', 'UcurrMean', 'CumCurrent',
                    'month']  # Include month for seasonality

# Add seasonal dummies if available
seasonal_features = [col for col in env_features_monthly.columns if col.startswith('Season_')]
monthly_features.extend(seasonal_features)

# Add annual context features (annual maxima assigned to each month)
annual_context_features = ['Hm0_max_annual', 'WindMax_annual', 'UcurrMax_annual']
available_annual = [f for f in annual_context_features if f in env_features_monthly.columns]
monthly_features.extend(available_annual)

# Remove any duplicates and unavailable features
model_features = [f for f in monthly_features if f in env_features_monthly.columns]
model_features = list(dict.fromkeys(model_features))  # Remove duplicates

X_monthly = env_features_monthly[model_features].values
y_monthly = env_features_monthly['Erosion_Label'].values
groups = env_features_monthly['monsoon_year'].values  # Group by year for proper CV

# Scale features
scaler = StandardScaler()
X_scaled_monthly = scaler.fit_transform(X_monthly)

# Also prepare annual data for comparison
model_features_annual = ['Hm0_max', 'Hm0_mean', 'CumWaveEnergy_annual', 'StormDays_wave_annual',
                         'WindMax', 'WindMean', 'WindStressMean',
                         'UcurrMax', 'UcurrMean', 'CumCurrent']
# Filter to available features
model_features_annual = [f for f in model_features_annual if f in env_features.columns]

# Fallback to basic features if some are missing
if len(model_features_annual) < 5:
    model_features_annual = ['Hm0_max', 'WindMax', 'UcurrMax', 'wave_intensity', 
                             'wind_intensity', 'current_intensity']
    model_features_annual = [f for f in model_features_annual if f in env_features.columns]

X_annual = env_features[model_features_annual].values
y_annual = env_features['Erosion_Label'].values

scaler_annual = StandardScaler()
X_scaled_annual = scaler_annual.fit_transform(X_annual)

print("="*60)
print("MODEL TRAINING DATA COMPARISON")
print("="*60)
print(f"\n📊 MONTHLY DATA (Recommended for Training):")
print(f"   Total samples: {len(y_monthly)}")
print(f"   Erosion months: {sum(y_monthly)} ({sum(y_monthly)/len(y_monthly)*100:.1f}%)")
print(f"   Stable months: {len(y_monthly)-sum(y_monthly)} ({(len(y_monthly)-sum(y_monthly))/len(y_monthly)*100:.1f}%)")
print(f"   Features: {len(model_features)}")
print(f"   Unique years (groups): {len(np.unique(groups))}")

print(f"\n📊 ANNUAL DATA (For Comparison):")
print(f"   Total samples: {len(y_annual)}")
print(f"   Erosion years: {sum(y_annual)} ({sum(y_annual)/len(y_annual)*100:.1f}%)")
print(f"   Stable years: {len(y_annual)-sum(y_annual)} ({(len(y_annual)-sum(y_annual))/len(y_annual)*100:.1f}%)")
print(f"   Features: {len(model_features_annual)}")

print(f"\n✅ Sample increase: {len(y_monthly)/len(y_annual):.1f}x more training data!")
print(f"\n⚠️ NOTE: Using GroupKFold CV to prevent data leakage between years")
print(f"   (All months from same year stay in same fold)")

print(f"\nMonthly Features: {model_features}")
print(f"Annual Features: {model_features_annual}")

MODEL TRAINING DATA COMPARISON

📊 MONTHLY DATA (Recommended for Training):
   Total samples: 300
   Erosion months: 84 (28.0%)
   Stable months: 216 (72.0%)
   Features: 18


   Unique years (groups): 25

📊 ANNUAL DATA (For Comparison):
   Total samples: 25
   Erosion years: 7 (28.0%)
   Stable years: 18 (72.0%)
   Features: 10

✅ Sample increase: 12.0x more training data!

⚠️ NOTE: Using GroupKFold CV to prevent data leakage between years
   (All months from same year stay in same fold)

Monthly Features: ['Hm0_max', 'Hm0_mean', 'CumWaveEnergy', 'StormDays_wave', 'WindMax', 'WindMean', 'WindStressMean', 'UcurrMax', 'UcurrMean', 'CumCurrent', 'month', 'Season_NE_Monsoon', 'Season_NE_Peak', 'Season_Pre_Monsoon', 'Season_SW_Monsoon', 'Hm0_max_annual', 'WindMax_annual', 'UcurrMax_annual']
Annual Features: ['Hm0_max', 'Hm0_mean', 'CumWaveEnergy_annual', 'StormDays_wave_annual', 'WindMax', 'WindMean', 'WindStressMean', 'UcurrMax', 'UcurrMean', 'CumCurrent']


In [20]:
# =============================================================================
# Section 6.2: Hidden Markov Model (HMM) for State Detection
# Using MONTHLY data for sequential state estimation
# =============================================================================

from hmmlearn.hmm import GaussianHMM

print("="*60)
print("HMM - OPTIMAL STATE SELECTION (Monthly Data)")
print("="*60)

# Find optimal number of hidden states using BIC
bic_scores = []
aic_scores = []
n_components_range = range(2, min(8, len(X_scaled_monthly)//30 + 1))

for n in n_components_range:
    hmm_temp = GaussianHMM(
        n_components=n, covariance_type='full',
        random_state=42, n_iter=200, tol=1e-4
    )
    hmm_temp.fit(X_scaled_monthly)
    log_likelihood = hmm_temp.score(X_scaled_monthly)
    n_params = n * n + n * X_scaled_monthly.shape[1] + n * X_scaled_monthly.shape[1] * (X_scaled_monthly.shape[1] + 1) // 2
    n_samples = len(X_scaled_monthly)
    bic = -2 * log_likelihood * n_samples + n_params * np.log(n_samples)
    aic = -2 * log_likelihood * n_samples + 2 * n_params
    bic_scores.append(bic)
    aic_scores.append(aic)
    print(f"  n_components={n}: BIC={bic:.1f}, AIC={aic:.1f}")

# Select optimal based on lowest BIC
optimal_n = list(n_components_range)[np.argmin(bic_scores)]
print(f"\n✓ Optimal number of hidden states (by BIC): {optimal_n}")

# Fit HMM with optimal states
n_states = optimal_n
hmm_model = GaussianHMM(
    n_components=n_states,
    covariance_type='full',
    random_state=42,
    n_iter=300,
    tol=1e-4
)
hmm_model.fit(X_scaled_monthly)

# Predict states for monthly data (Viterbi decoding)
env_features_monthly['HMM_State'] = hmm_model.predict(X_scaled_monthly)
hmm_probs = hmm_model.predict_proba(X_scaled_monthly)

for i in range(n_states):
    env_features_monthly[f'HMM_Prob_State{i}'] = hmm_probs[:, i]

# HMM-specific: transition matrix
transition_matrix = hmm_model.transmat_

print("\n" + "="*60)
print("HIDDEN MARKOV MODEL - STATE DETECTION")
print("="*60)
print(f"Number of hidden states: {n_states}")
print(f"Log-likelihood: {hmm_model.score(X_scaled_monthly):.3f}")

# Calculate BIC/AIC for the optimal model
_ll = hmm_model.score(X_scaled_monthly) * len(X_scaled_monthly)
_n_params = n_states * n_states + n_states * X_scaled_monthly.shape[1] + n_states * X_scaled_monthly.shape[1] * (X_scaled_monthly.shape[1] + 1) // 2
hmm_bic = -2 * _ll + _n_params * np.log(len(X_scaled_monthly))
hmm_aic = -2 * _ll + 2 * _n_params
print(f"AIC: {hmm_aic:.3f}")
print(f"BIC: {hmm_bic:.3f}")
hmm_converged = hmm_model.monitor_.converged
print(f"Converged: {hmm_converged}")

print(f"\nTransition Matrix:")
tm_df = pd.DataFrame(transition_matrix,
                      index=[f'From S{i}' for i in range(n_states)],
                      columns=[f'To S{i}' for i in range(n_states)])
display(tm_df.round(3))

# State means (in original scale)
state_means = scaler.inverse_transform(hmm_model.means_)
state_means_df = pd.DataFrame(state_means, columns=model_features)
state_means_df.index = [f'State_{i}' for i in range(n_states)]

print(f"\nState Centroids (showing first 6 features):")
display(state_means_df.iloc[:, :6].round(3))

# Determine which state corresponds to erosion (highest erosion rate)
state_erosion_rate = env_features_monthly.groupby('HMM_State')['Erosion_Label'].mean()
erosion_state = state_erosion_rate.idxmax()

print(f"\nErosion Rate by State:")
for state, rate in state_erosion_rate.items():
    label = '⚠ EROSION STATE' if state == erosion_state else 'Normal/Stable'
    print(f"  State {state}: {rate*100:.1f}% erosion months → {label}")

# Regime stability: count state switches per monsoon year
state_switches = []
for year in env_features_monthly['monsoon_year'].unique():
    year_states = env_features_monthly[env_features_monthly['monsoon_year'] == year]['HMM_State'].values
    switches = np.sum(np.diff(year_states) != 0)
    state_switches.append(switches)

regime_stability = pd.Series(state_switches)
print(f"\nRegime Stability (state switches per year):")
print(f"  Mean: {regime_stability.mean():.1f}, Std: {regime_stability.std():.1f}")
print(f"  Min: {regime_stability.min()}, Max: {regime_stability.max()}")

# Final-state dominance: distribution of final states in erosion years
erosion_year_labels = env_features_monthly.groupby('monsoon_year')['Erosion_Label'].max()
erosion_years = erosion_year_labels[erosion_year_labels == 1].index
final_states_erosion = []
for year in erosion_years:
    year_data = env_features_monthly[env_features_monthly['monsoon_year'] == year]
    if len(year_data) > 0:
        final_states_erosion.append(year_data.iloc[-1]['HMM_State'])

final_state_dist = pd.Series(final_states_erosion).value_counts(normalize=True).sort_index()
print(f"\nFinal-State Dominance (erosion years):")
for state, pct in final_state_dist.items():
    print(f"  State {int(state)}: {pct*100:.0f}%")

# Plot state distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# State counts
ax1 = axes[0]
state_counts = env_features_monthly['HMM_State'].value_counts().sort_index()
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, n_states))
bars = ax1.bar(state_counts.index, state_counts.values, color=colors)
ax1.set_xlabel('HMM State')
ax1.set_ylabel('Count (Months)')
ax1.set_title('Monthly State Distribution')
for i, (count, rate) in enumerate(zip(state_counts.values, state_erosion_rate)):
    ax1.annotate(f'{rate*100:.0f}% erosion', (i, count), ha='center', va='bottom')

# State erosion rates
ax2 = axes[1]
ax2.bar(state_erosion_rate.index, state_erosion_rate.values * 100, 
        color=['red' if s == erosion_state else 'green' for s in state_erosion_rate.index])
ax2.axhline(y=50, color='gray', linestyle='--', alpha=0.5)
ax2.set_xlabel('HMM State')
ax2.set_ylabel('Erosion Rate (%)')
ax2.set_title('Erosion Rate by HMM State')

plt.tight_layout()
plt.show()

Model is not converging.  Current: -1063.1798352008693 is not greater than 1819.3953039479159. Delta is -2882.575139148785


HMM - OPTIMAL STATE SELECTION (Monthly Data)
  n_components=2: BIC=444321.6, AIC=442906.7


  n_components=3: BIC=-1754454.5, AIC=-1756587.9


  n_components=4: BIC=-3450345.6, AIC=-3453204.9


  n_components=5: BIC=-4309257.7, AIC=-4312850.4


...


HIDDEN MARKOV MODEL - STATE DETECTION
Number of hidden states: 7
Log-likelihood: 7475.535
AIC: -4482577.072
BIC: -4477495.483
Converged: True

Transition Matrix:


,To S0,To S1,To S2,To S3,To S4,To S5,To S6
From S0,0.0,0.000,0.000,0.000,1.00,0.000,0.000
From S1,0.0,0.000,0.000,0.000,1.00,0.000,0.000
From S2,0.0,0.000,0.000,1.000,0.00,0.000,0.000
From S3,0.0,0.000,0.169,0.662,0.00,0.000,0.169
From S4,0.0,0.000,0.000,0.510,0.49,0.000,0.000
From S5,1.0,0.000,0.000,0.000,0.00,0.000,0.000
From S6,0.0,0.432,0.000,0.000,0.00,0.136,0.432



State Centroids (showing first 6 features):


,Hm0_max,Hm0_mean,CumWaveEnergy,StormDays_wave,WindMax,WindMean
State_0,1.990,0.906,2641.143,2.500,4.243,4.243
State_1,1.246,0.786,2068.761,0.000,3.899,3.899
State_2,2.378,1.758,9053.509,38.400,6.230,6.230
State_3,2.050,1.352,6117.707,19.530,4.971,4.971
State_4,1.125,0.739,1829.252,0.000,3.455,3.455
State_5,1.962,0.990,2906.780,2.333,4.378,4.378
State_6,1.824,1.062,3588.178,3.614,4.274,4.274



Erosion Rate by State:
  State 0: 50.0% erosion months → ⚠ EROSION STATE
  State 1: 21.1% erosion months → Normal/Stable
  State 2: 28.0% erosion months → Normal/Stable
  State 3: 28.2% erosion months → Normal/Stable
  State 4: 27.5% erosion months → Normal/Stable
  State 5: 50.0% erosion months → Normal/Stable
  State 6: 25.0% erosion months → Normal/Stable

Regime Stability (state switches per year):
  Mean: 6.3, Std: 0.5
  Min: 6, Max: 8

Final-State Dominance (erosion years):
  State 3: 100%


In [21]:
# =============================================================================
# Section 6.3: HMM Threshold Extraction (Per-Variable Ranges)
# =============================================================================

# Identify the erosion state (highest erosion rate)
erosion_state = state_erosion_rate.idxmax()
erosion_centroid = state_means_df.loc[f'State_{erosion_state}']

# Normal states = all non-erosion states
normal_states = [i for i in range(n_states) if i != erosion_state]
normal_centroids = state_means_df.loc[[f'State_{i}' for i in normal_states]]

print("="*60)
print("HMM-BASED EROSION THRESHOLDS (Per Variable)")
print("="*60)
print(f"Erosion State: State {erosion_state} (erosion rate: {state_erosion_rate[erosion_state]*100:.1f}%)")
print(f"Normal States: {normal_states}\n")

# Key physical drivers (skip month and season dummies)
hmm_key_drivers = ['Hm0_max', 'Hm0_mean', 'CumWaveEnergy', 'StormDays_wave',
                   'WindMax', 'WindMean', 'WindStressMean',
                   'UcurrMax', 'UcurrMean', 'CumCurrent']
hmm_key_drivers = [d for d in hmm_key_drivers if d in model_features]

# For each driver, compute threshold range from HMM state statistics
# Threshold = decision boundary region between erosion and normal state distributions
hmm_thresholds = {}

for feature in hmm_key_drivers:
    erosion_val = erosion_centroid[feature]
    normal_val = normal_centroids[feature].mean()
    
    # State standard deviations (from covariance matrix, in original scale)
    feat_idx = model_features.index(feature)
    erosion_std_scaled = np.sqrt(hmm_model.covars_[erosion_state][feat_idx, feat_idx])
    erosion_std = erosion_std_scaled * scaler.scale_[feat_idx]
    
    normal_stds = []
    for ns in normal_states:
        ns_std_scaled = np.sqrt(hmm_model.covars_[ns][feat_idx, feat_idx])
        normal_stds.append(ns_std_scaled * scaler.scale_[feat_idx])
    normal_std = np.mean(normal_stds)
    
    # Threshold midpoint = weighted average of erosion and normal centroids
    midpoint = (erosion_val + normal_val) / 2
    
    # Threshold range: midpoint ± half the gap between centroids
    half_gap = abs(erosion_val - normal_val) / 4  # conservative range
    threshold_lower = midpoint - half_gap
    threshold_upper = midpoint + half_gap
    
    direction = "≥" if erosion_val > normal_val else "≤"
    
    hmm_thresholds[feature] = {
        'threshold': midpoint,
        'threshold_lower': threshold_lower,
        'threshold_upper': threshold_upper,
        'direction': direction,
        'erosion_value': erosion_val,
        'normal_value': normal_val,
        'erosion_std': erosion_std,
        'normal_std': normal_std
    }

# Display as table
hmm_threshold_rows = []
for feat in hmm_key_drivers:
    t = hmm_thresholds[feat]
    hmm_threshold_rows.append({
        'Variable': feat,
        'Direction': t['direction'],
        'Threshold (midpoint)': round(t['threshold'], 4),
        'Range Lower': round(t['threshold_lower'], 4),
        'Range Upper': round(t['threshold_upper'], 4),
        'Erosion State Mean': round(t['erosion_value'], 4),
        'Normal State Mean': round(t['normal_value'], 4)
    })

hmm_threshold_table = pd.DataFrame(hmm_threshold_rows)
print("\nHMM Threshold Table (Original Scale):")
display(hmm_threshold_table)

# Identify top discriminating features
threshold_diff = {k: abs(v['erosion_value'] - v['normal_value']) 
                  for k, v in hmm_thresholds.items()}
top_hmm_features = sorted(threshold_diff.items(), key=lambda x: x[1], reverse=True)[:5]

print(f"\nTop 5 Discriminating Features (HMM):")
for feat, diff in top_hmm_features:
    print(f"  {feat}: gap = {diff:.4f} between erosion/normal states")

HMM-BASED EROSION THRESHOLDS (Per Variable)
Erosion State: State 0 (erosion rate: 50.0%)
Normal States: [1, 2, 3, 4, 5, 6]


HMM Threshold Table (Original Scale):


,Variable,Direction,Threshold (midpoint),Range Lower,Range Upper,Erosion State Mean,Normal State Mean
0,Hm0_max,≥,1.8771,1.8207,1.9336,1.9900,1.7642
1,Hm0_mean,≤,1.0102,0.9581,1.0622,0.9061,1.1143
2,CumWaveEnergy,≤,3450.9203,3046.0315,3855.8090,2641.1428,4260.6978
3,StormDays_wave,≤,6.5731,4.5365,8.6096,2.5000,10.6462
4,WindMax,≤,4.3889,4.3161,4.4617,4.2433,4.5345
5,WindMean,≤,4.3889,4.3161,4.4617,4.2433,4.5345
6,WindStressMean,≥,0.0422,0.0408,0.0436,0.0450,0.0394
7,UcurrMax,≥,0.3607,0.3606,0.3608,0.3610,0.3605
8,UcurrMean,≤,0.1646,0.1640,0.1651,0.1634,0.1657
9,CumCurrent,≥,5.0426,5.0308,5.0543,5.0661,5.0190



Top 5 Discriminating Features (HMM):
  CumWaveEnergy: gap = 1619.5550 between erosion/normal states
  StormDays_wave: gap = 8.1462 between erosion/normal states
  WindMax: gap = 0.2912 between erosion/normal states
  WindMean: gap = 0.2912 between erosion/normal states
  Hm0_max: gap = 0.2258 between erosion/normal states


In [22]:
# =============================================================================
# Section 6.4: Random Forest Model with GroupKFold Cross-Validation
# Using MONTHLY data with proper group-based CV to prevent data leakage
# =============================================================================

from sklearn.model_selection import RandomizedSearchCV, GroupKFold, cross_val_predict

print("="*60)
print("RANDOM FOREST MODEL - MONTHLY DATA WITH GROUP CV")
print("="*60)

# Define parameter grid
rf_param_dist = {
    'n_estimators': [100, 150, 200, 300],
    'max_depth': [3, 4, 5, 6, 8, None],
    'min_samples_split': [2, 5, 10, 15],
    'min_samples_leaf': [1, 2, 4, 6],
    'max_features': ['sqrt', 'log2', 0.3, 0.5],
    'class_weight': ['balanced', 'balanced_subsample']
}

# Use GroupKFold to ensure all months from same year stay together
# This prevents data leakage (can't use future months to predict past)
n_splits = min(5, len(np.unique(groups)))
group_cv = GroupKFold(n_splits=n_splits)

# Base model
rf_base = RandomForestClassifier(random_state=42, oob_score=True, n_jobs=-1)

# Randomized search with GroupKFold
rf_search = RandomizedSearchCV(
    rf_base, 
    rf_param_dist, 
    n_iter=50,
    cv=group_cv,
    scoring='f1',
    n_jobs=-1, 
    random_state=42,
    verbose=0
)

print(f"Using GroupKFold with {n_splits} splits (groups = years)")
print("Running hyperparameter search...")
rf_search.fit(X_scaled_monthly, y_monthly, groups=groups)

print(f"\n✓ Best Parameters Found:")
for param, value in rf_search.best_params_.items():
    print(f"    {param}: {value}")
print(f"\n✓ Best CV F1 Score: {rf_search.best_score_:.3f}")

# Get best model
rf_model = rf_search.best_estimator_

# Get cross-validated predictions (each month predicted when its year was in test set)
y_pred_rf_monthly = cross_val_predict(rf_model, X_scaled_monthly, y_monthly, 
                                       cv=group_cv, groups=groups, method='predict')
y_prob_rf_monthly = cross_val_predict(rf_model, X_scaled_monthly, y_monthly, 
                                       cv=group_cv, groups=groups, method='predict_proba')[:, 1]

# Calculate metrics
print("\n" + "-"*40)
print("Cross-Validated Performance (GroupKFold):")
print(f"  Accuracy: {accuracy_score(y_monthly, y_pred_rf_monthly):.3f}")
print(f"  Precision: {precision_score(y_monthly, y_pred_rf_monthly, zero_division=0):.3f}")
print(f"  Recall: {recall_score(y_monthly, y_pred_rf_monthly, zero_division=0):.3f}")
print(f"  F1-Score: {f1_score(y_monthly, y_pred_rf_monthly, zero_division=0):.3f}")
print(f"  ROC-AUC: {roc_auc_score(y_monthly, y_prob_rf_monthly):.3f}")

# Refit on full data for feature importance
rf_model.fit(X_scaled_monthly, y_monthly)

if hasattr(rf_model, 'oob_score_'):
    print(f"\nOut-of-Bag Score: {rf_model.oob_score_:.3f}")

# Store predictions for comparison
y_pred_rf = y_pred_rf_monthly
y_prob_rf = y_prob_rf_monthly
y = y_monthly  # Update reference

# Feature importance
feature_importance = pd.DataFrame({
    'Feature': model_features,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print(f"\nTop 10 Feature Importance:")
display(feature_importance.head(10))

# Plot feature importance
fig, ax = plt.subplots(figsize=(10, 6))
top_n = min(15, len(feature_importance))
top_features_df = feature_importance.head(top_n)
ax.barh(range(top_n), top_features_df['Importance'].values[::-1], color='steelblue')
ax.set_yticks(range(top_n))
ax.set_yticklabels(top_features_df['Feature'].values[::-1])
ax.set_xlabel('Importance')
ax.set_title('Random Forest Feature Importance (Monthly Model)')
plt.tight_layout()
plt.show()

RANDOM FOREST MODEL - MONTHLY DATA WITH GROUP CV
Using GroupKFold with 5 splits (groups = years)
Running hyperparameter search...



✓ Best Parameters Found:
    n_estimators: 100
    min_samples_split: 10
    min_samples_leaf: 4
    max_features: log2
    max_depth: 3
    class_weight: balanced_subsample

✓ Best CV F1 Score: 0.375



----------------------------------------
Cross-Validated Performance (GroupKFold):
  Accuracy: 0.750
  Precision: 0.600
  Recall: 0.321
  F1-Score: 0.419
  ROC-AUC: 0.640



Out-of-Bag Score: 0.900

Top 10 Feature Importance:


,Feature,Importance
17,UcurrMax_annual,0.286208
15,Hm0_max_annual,0.278859
16,WindMax_annual,0.146797
6,WindStressMean,0.058899
5,WindMean,0.037160
7,UcurrMax,0.031625
0,Hm0_max,0.030627
2,CumWaveEnergy,0.027967
1,Hm0_mean,0.024518
8,UcurrMean,0.022135


In [23]:
# =============================================================================
# Section 6.5: Random Forest Threshold Extraction (Per-Variable Ranges)
# =============================================================================

def extract_tree_thresholds(rf_model, feature_names, scaler, top_n=5):
    """Extract thresholds from Random Forest decision trees and convert to original scale."""
    all_thresholds = {feat: [] for feat in feature_names}
    
    for tree in rf_model.estimators_:
        tree_ = tree.tree_
        feature_idx = tree_.feature
        threshold = tree_.threshold
        
        for node_id in range(tree_.node_count):
            if tree_.children_left[node_id] != tree_.children_right[node_id]:  # Not a leaf
                feat_idx = feature_idx[node_id]
                feat_name = feature_names[feat_idx]
                # Convert from scaled to original
                original_thresh = threshold[node_id] * scaler.scale_[feat_idx] + scaler.mean_[feat_idx]
                all_thresholds[feat_name].append(original_thresh)
    
    return all_thresholds

# Extract all thresholds in original scale
rf_all_thresholds = extract_tree_thresholds(rf_model, model_features, scaler)

# Key drivers
rf_key_drivers = ['Hm0_max', 'Hm0_mean', 'CumWaveEnergy', 'StormDays_wave',
                  'WindMax', 'WindMean', 'WindStressMean',
                  'UcurrMax', 'UcurrMean', 'CumCurrent']
rf_key_drivers = [d for d in rf_key_drivers if d in model_features]

print("="*60)
print("RANDOM FOREST THRESHOLD EXTRACTION (Per Variable)")
print("="*60)

rf_thresholds = {}
rf_threshold_rows = []

for feature in rf_key_drivers:
    thresholds = rf_all_thresholds.get(feature, [])
    if not thresholds:
        continue
    
    thresholds = np.array(thresholds)
    
    # Use percentiles to define the threshold range
    median_thresh = np.median(thresholds)
    q25_thresh = np.percentile(thresholds, 25)
    q75_thresh = np.percentile(thresholds, 75)
    mean_thresh = np.mean(thresholds)
    
    # Get feature importance for this variable
    imp = feature_importance[feature_importance['Feature'] == feature]['Importance'].values
    importance = imp[0] if len(imp) > 0 else 0
    
    # Determine direction: compare erosion-class vs stable-class means
    erosion_mean = env_features_monthly[env_features_monthly['Erosion_Label'] == 1][feature].mean()
    stable_mean = env_features_monthly[env_features_monthly['Erosion_Label'] == 0][feature].mean()
    direction = "≥" if erosion_mean > stable_mean else "≤"
    
    rf_thresholds[feature] = {
        'threshold': median_thresh,
        'threshold_lower': q25_thresh,
        'threshold_upper': q75_thresh,
        'mean_threshold': mean_thresh,
        'direction': direction,
        'n_splits': len(thresholds),
        'importance': importance,
        'erosion_mean': erosion_mean,
        'stable_mean': stable_mean
    }
    
    rf_threshold_rows.append({
        'Variable': feature,
        'Direction': direction,
        'Threshold (median)': round(median_thresh, 4),
        'Range Lower (Q25)': round(q25_thresh, 4),
        'Range Upper (Q75)': round(q75_thresh, 4),
        'N Splits': len(thresholds),
        'Importance': round(importance, 4)
    })

rf_threshold_table = pd.DataFrame(rf_threshold_rows)
rf_threshold_table = rf_threshold_table.sort_values('Importance', ascending=False).reset_index(drop=True)

print("\nRandom Forest Threshold Table (Original Scale):")
display(rf_threshold_table)

print(f"\nTotal decision splits analysed: {sum(len(v) for v in rf_all_thresholds.values())}")
print(f"Key drivers with thresholds: {len(rf_thresholds)}")

RANDOM FOREST THRESHOLD EXTRACTION (Per Variable)

Random Forest Threshold Table (Original Scale):


,Variable,Direction,Threshold (median),Range Lower (Q25),Range Upper (Q75),N Splits,Importance
0,WindStressMean,≥,0.0350,0.0350,0.0350,24,0.0589
1,WindMean,≤,3.6300,3.1550,4.2800,29,0.0372
2,UcurrMax,≥,0.3297,0.2494,0.3788,26,0.0316
3,Hm0_max,≥,2.4550,2.1175,2.7075,27,0.0306
4,CumWaveEnergy,≥,4056.0627,1857.0243,9437.9842,31,0.0280
5,Hm0_mean,≥,0.7895,0.7397,1.3749,27,0.0245
6,UcurrMean,≤,0.1592,0.1276,0.2010,21,0.0221
7,WindMax,≤,3.8650,3.0150,4.9100,21,0.0203
8,CumCurrent,≤,3.6787,3.2301,4.7335,27,0.0200
9,StormDays_wave,≥,11.7500,5.6250,25.7500,16,0.0107



Total decision splits analysed: 517
Key drivers with thresholds: 10


In [24]:
# =============================================================================
# Section 6.6: XGBoost Model with GroupKFold Cross-Validation
# Using MONTHLY data for better generalization
# =============================================================================

print("="*60)
print("XGBOOST MODEL - MONTHLY DATA WITH GROUP CV")
print("="*60)

# Create DataFrames with column names so XGBoost stores feature names in booster
X_df_monthly = pd.DataFrame(X_scaled_monthly, columns=model_features)

# Define parameter grid
xgb_param_dist = {
    'n_estimators': [100, 150, 200, 300],
    'max_depth': [2, 3, 4, 5, 6],
    'learning_rate': [0.01, 0.05, 0.1, 0.15],
    'min_child_weight': [1, 2, 3, 5],
    'subsample': [0.6, 0.7, 0.8, 0.9],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9],
    'gamma': [0, 0.1, 0.2, 0.3],
    'reg_alpha': [0, 0.01, 0.1],
    'reg_lambda': [1, 1.5, 2],
    'scale_pos_weight': [1, sum(y_monthly==0)/max(sum(y_monthly==1), 1)]
}

# Base XGBoost model
xgb_base = xgb.XGBClassifier(
    random_state=42,
    eval_metric='logloss',
    verbosity=0,
    n_jobs=-1
)

# Randomized search with GroupKFold
xgb_search = RandomizedSearchCV(
    xgb_base, 
    xgb_param_dist, 
    n_iter=50,
    cv=group_cv,
    scoring='f1',
    n_jobs=-1, 
    random_state=42,
    verbose=0
)

print(f"Using GroupKFold with {n_splits} splits")
print("Running hyperparameter search...")
xgb_search.fit(X_df_monthly, y_monthly, groups=groups)

print(f"\n✓ Best Parameters Found:")
for param, value in xgb_search.best_params_.items():
    print(f"    {param}: {value}")
print(f"\n✓ Best CV F1 Score: {xgb_search.best_score_:.3f}")

# Get best model
xgb_model = xgb_search.best_estimator_

# Get cross-validated predictions
y_pred_xgb_monthly = cross_val_predict(xgb_model, X_df_monthly, y_monthly, 
                                        cv=group_cv, groups=groups, method='predict')
y_prob_xgb_monthly = cross_val_predict(xgb_model, X_df_monthly, y_monthly, 
                                        cv=group_cv, groups=groups, method='predict_proba')[:, 1]

# Calculate metrics
print("\n" + "-"*40)
print("Cross-Validated Performance (GroupKFold):")
print(f"  Accuracy: {accuracy_score(y_monthly, y_pred_xgb_monthly):.3f}")
print(f"  Precision: {precision_score(y_monthly, y_pred_xgb_monthly, zero_division=0):.3f}")
print(f"  Recall: {recall_score(y_monthly, y_pred_xgb_monthly, zero_division=0):.3f}")
print(f"  F1-Score: {f1_score(y_monthly, y_pred_xgb_monthly, zero_division=0):.3f}")
print(f"  ROC-AUC: {roc_auc_score(y_monthly, y_prob_xgb_monthly):.3f}")

# Refit on full data (with DataFrame to preserve feature names)
xgb_model.fit(X_df_monthly, y_monthly, verbose=False)

# Store predictions
y_pred_xgb = y_pred_xgb_monthly
y_prob_xgb = y_prob_xgb_monthly

# Feature importance
xgb_importance = pd.DataFrame({
    'Feature': model_features,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False)

print(f"\nTop 10 XGBoost Feature Importance:")
display(xgb_importance.head(10))

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
top_n = min(15, len(xgb_importance))
top_xgb = xgb_importance.head(top_n)
ax.barh(range(top_n), top_xgb['Importance'].values[::-1], color='coral')
ax.set_yticks(range(top_n))
ax.set_yticklabels(top_xgb['Feature'].values[::-1])
ax.set_xlabel('Importance (Gain)')
ax.set_title('XGBoost Feature Importance (Monthly Model)')
plt.tight_layout()
plt.show()

XGBOOST MODEL - MONTHLY DATA WITH GROUP CV
Using GroupKFold with 5 splits
Running hyperparameter search...



✓ Best Parameters Found:
    subsample: 0.7
    scale_pos_weight: 2.5714285714285716
    reg_lambda: 2
    reg_alpha: 0.01
    n_estimators: 200
    min_child_weight: 5
    max_depth: 5
    learning_rate: 0.01
    gamma: 0.2
    colsample_bytree: 0.9

✓ Best CV F1 Score: 0.633



----------------------------------------
Cross-Validated Performance (GroupKFold):
  Accuracy: 0.800
  Precision: 0.667
  Recall: 0.571
  F1-Score: 0.615
  ROC-AUC: 0.663



Top 10 XGBoost Feature Importance:


,Feature,Importance
15,Hm0_max_annual,0.319242
17,UcurrMax_annual,0.298836
16,WindMax_annual,0.195099
2,CumWaveEnergy,0.035160
8,UcurrMean,0.030607
9,CumCurrent,0.030337
4,WindMax,0.029181
0,Hm0_max,0.020602
7,UcurrMax,0.019728
1,Hm0_mean,0.015768


In [25]:
# =============================================================================
# Section 6.7: XGBoost Threshold Extraction + SHAP Analysis
# =============================================================================

# --- A. Extract split thresholds from XGBoost trees ---

def extract_xgb_thresholds(xgb_model, feature_names, scaler):
    """Extract thresholds from XGBoost booster trees and convert to original scale.
    Handles both named features and default f0/f1/... naming convention."""
    all_thresholds = {feat: [] for feat in feature_names}
    
    booster = xgb_model.get_booster()
    trees_df = booster.trees_to_dataframe()
    
    # Build mapping: f0 -> feature_names[0], f1 -> feature_names[1], ...
    idx_to_name = {f'f{i}': name for i, name in enumerate(feature_names)}
    name_set = set(feature_names)
    
    # Filter to split nodes only
    splits = trees_df[trees_df['Feature'] != 'Leaf'].copy()
    
    matched = 0
    for _, row in splits.iterrows():
        raw_feat = str(row['Feature'])
        # Resolve to actual feature name (handle both conventions)
        if raw_feat in name_set:
            feat_name = raw_feat
        elif raw_feat in idx_to_name:
            feat_name = idx_to_name[raw_feat]
        else:
            continue
        
        feat_idx = feature_names.index(feat_name)
        original_thresh = row['Split'] * scaler.scale_[feat_idx] + scaler.mean_[feat_idx]
        all_thresholds[feat_name].append(original_thresh)
        matched += 1
    
    print(f"  Matched {matched}/{len(splits)} split nodes to features")
    return all_thresholds

xgb_all_thresholds = extract_xgb_thresholds(xgb_model, model_features, scaler)

# Key drivers
xgb_key_drivers = ['Hm0_max', 'Hm0_mean', 'CumWaveEnergy', 'StormDays_wave',
                   'WindMax', 'WindMean', 'WindStressMean',
                   'UcurrMax', 'UcurrMean', 'CumCurrent']
xgb_key_drivers = [d for d in xgb_key_drivers if d in model_features]

xgb_thresholds = {}
xgb_threshold_rows = []

for feature in xgb_key_drivers:
    thresholds = xgb_all_thresholds.get(feature, [])
    if not thresholds:
        continue
    
    thresholds = np.array(thresholds)
    
    median_thresh = np.median(thresholds)
    q25_thresh = np.percentile(thresholds, 25)
    q75_thresh = np.percentile(thresholds, 75)
    
    # Get XGBoost feature importance
    imp = xgb_importance[xgb_importance['Feature'] == feature]['Importance'].values
    importance = imp[0] if len(imp) > 0 else 0
    
    erosion_mean = env_features_monthly[env_features_monthly['Erosion_Label'] == 1][feature].mean()
    stable_mean = env_features_monthly[env_features_monthly['Erosion_Label'] == 0][feature].mean()
    direction = "≥" if erosion_mean > stable_mean else "≤"
    
    xgb_thresholds[feature] = {
        'threshold': median_thresh,
        'threshold_lower': q25_thresh,
        'threshold_upper': q75_thresh,
        'direction': direction,
        'n_splits': len(thresholds),
        'importance': importance,
        'erosion_mean': erosion_mean,
        'stable_mean': stable_mean
    }
    
    xgb_threshold_rows.append({
        'Variable': feature,
        'Direction': direction,
        'Threshold (median)': round(median_thresh, 4),
        'Range Lower (Q25)': round(q25_thresh, 4),
        'Range Upper (Q75)': round(q75_thresh, 4),
        'N Splits': len(thresholds),
        'Importance': round(importance, 4)
    })

xgb_threshold_table = pd.DataFrame(xgb_threshold_rows)
if not xgb_threshold_table.empty:
    xgb_threshold_table = xgb_threshold_table.sort_values('Importance', ascending=False).reset_index(drop=True)

print("="*60)
print("XGBOOST THRESHOLD EXTRACTION (Per Variable)")
print("="*60)
total_splits = sum(len(v) for v in xgb_all_thresholds.values())
print(f"\nTotal splits extracted: {total_splits}")
print(f"Variables with thresholds: {len(xgb_thresholds)}")
print("\nXGBoost Threshold Table (Original Scale):")
display(xgb_threshold_table)

# --- B. SHAP Analysis ---
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_df_monthly)
feature_names_short = model_features

print("\n" + "="*60)
print("SHAP ANALYSIS - XGBoost Feature Contributions")
print("="*60)

# SHAP Summary Plots
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

plt.sca(axes[0])
shap.summary_plot(shap_values, X_df_monthly, feature_names=feature_names_short, 
                  show=False, plot_size=None)
axes[0].set_title('SHAP Summary Plot')

plt.sca(axes[1])
shap.summary_plot(shap_values, X_df_monthly, feature_names=feature_names_short,
                  plot_type='bar', show=False, plot_size=None)
axes[1].set_title('Mean |SHAP Value|')

plt.tight_layout()
plt.show()

# Calculate mean absolute SHAP values
mean_shap = np.abs(shap_values).mean(axis=0)
shap_importance = pd.DataFrame({
    'Feature': model_features,
    'Mean_SHAP': mean_shap
}).sort_values('Mean_SHAP', ascending=False)

print("\nSHAP Feature Importance:")
display(shap_importance)

  Matched 951/951 split nodes to features
XGBOOST THRESHOLD EXTRACTION (Per Variable)

Total splits extracted: 951
Variables with thresholds: 8

XGBoost Threshold Table (Original Scale):


,Variable,Direction,Threshold (median),Range Lower (Q25),Range Upper (Q75),N Splits,Importance
0,CumWaveEnergy,≥,3224.1317,2241.7714,4333.5730,4,0.0352
1,UcurrMean,≤,0.1403,0.1347,0.1410,5,0.0306
2,CumCurrent,≤,4.9342,4.9342,4.9342,3,0.0303
3,WindMax,≤,3.9600,3.8550,4.0650,7,0.0292
4,Hm0_max,≥,1.9300,1.6900,2.1400,10,0.0206
5,UcurrMax,≥,0.3313,0.3035,0.3376,14,0.0197
6,Hm0_mean,≥,1.0152,0.9866,1.1541,3,0.0158
7,StormDays_wave,≥,5.0000,5.0000,5.0000,1,0.0054



SHAP ANALYSIS - XGBoost Feature Contributions



SHAP Feature Importance:


,Feature,Mean_SHAP
15,Hm0_max_annual,1.072443
17,UcurrMax_annual,0.590316
16,WindMax_annual,0.387695
7,UcurrMax,0.005948
0,Hm0_max,0.003949
8,UcurrMean,0.003356
4,WindMax,0.002494
9,CumCurrent,0.002141
2,CumWaveEnergy,0.001606
1,Hm0_mean,0.001380


## Section 7: Model Comparison and Evaluation

In [26]:
# =============================================================================
# Section 7.1: Model Performance Comparison
# =============================================================================

# HMM prediction: erosion state = erosion
hmm_pred_monthly = (env_features_monthly['HMM_State'] == erosion_state).astype(int)
hmm_prob_erosion = env_features_monthly[f'HMM_Prob_State{erosion_state}']

hmm_accuracy = accuracy_score(y_monthly, hmm_pred_monthly)
hmm_f1 = f1_score(y_monthly, hmm_pred_monthly, zero_division=0)
hmm_auc = roc_auc_score(y_monthly, hmm_prob_erosion)
hmm_prec = precision_score(y_monthly, hmm_pred_monthly, zero_division=0)
hmm_rec = recall_score(y_monthly, hmm_pred_monthly, zero_division=0)

rf_accuracy = accuracy_score(y_monthly, y_pred_rf)
rf_f1 = f1_score(y_monthly, y_pred_rf, zero_division=0)
rf_auc = roc_auc_score(y_monthly, y_prob_rf)
rf_prec = precision_score(y_monthly, y_pred_rf, zero_division=0)
rf_rec = recall_score(y_monthly, y_pred_rf, zero_division=0)

xgb_accuracy = accuracy_score(y_monthly, y_pred_xgb)
xgb_f1 = f1_score(y_monthly, y_pred_xgb, zero_division=0)
xgb_auc = roc_auc_score(y_monthly, y_prob_xgb)
xgb_prec = precision_score(y_monthly, y_pred_xgb, zero_division=0)
xgb_rec = recall_score(y_monthly, y_pred_xgb, zero_division=0)

model_comparison = pd.DataFrame({
    'Model': ['HMM', 'Random Forest', 'XGBoost'],
    'Accuracy': [hmm_accuracy, rf_accuracy, xgb_accuracy],
    'Precision': [hmm_prec, rf_prec, xgb_prec],
    'Recall': [hmm_rec, rf_rec, xgb_rec],
    'F1_Score': [hmm_f1, rf_f1, xgb_f1],
    'ROC_AUC': [hmm_auc, rf_auc, xgb_auc]
}).round(3)

print("="*70)
print("MODEL PERFORMANCE COMPARISON (Monthly Data, GroupKFold CV)")
print("="*70)
display(model_comparison)

# Confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, preds, title, cmap in [
    (axes[0], hmm_pred_monthly, f'HMM (Acc={hmm_accuracy:.3f})', 'Blues'),
    (axes[1], y_pred_rf, f'Random Forest (Acc={rf_accuracy:.3f})', 'Greens'),
    (axes[2], y_pred_xgb, f'XGBoost (Acc={xgb_accuracy:.3f})', 'Oranges')
]:
    cm = confusion_matrix(y_monthly, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=ax,
                xticklabels=['Stable', 'Erosion'], yticklabels=['Stable', 'Erosion'])
    ax.set_title(title)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.show()

# ROC Curves
fig, ax = plt.subplots(figsize=(10, 8))

fpr_hmm, tpr_hmm, _ = roc_curve(y_monthly, hmm_prob_erosion)
ax.plot(fpr_hmm, tpr_hmm, label=f'HMM (AUC={hmm_auc:.3f})', linewidth=2, color='blue')

fpr_rf, tpr_rf, _ = roc_curve(y_monthly, y_prob_rf)
ax.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC={rf_auc:.3f})', linewidth=2, color='green')

fpr_xgb, tpr_xgb, _ = roc_curve(y_monthly, y_prob_xgb)
ax.plot(fpr_xgb, tpr_xgb, label=f'XGBoost (AUC={xgb_auc:.3f})', linewidth=2, color='orange')

ax.plot([0, 1], [0, 1], 'k--', label='Random', alpha=0.5)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves - All Models (GroupKFold CV)')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# =============================================================================
# Section 7.2: Per-Variable Threshold Comparison Across Models
# =============================================================================

print("\n" + "="*70)
print("THRESHOLD COMPARISON ACROSS ALL THREE MODELS")
print("="*70)

# Collect all drivers present in all three models
all_drivers = sorted(set(hmm_thresholds.keys()) & set(rf_thresholds.keys()) & set(xgb_thresholds.keys()))

comparison_rows = []
for driver in all_drivers:
    h = hmm_thresholds[driver]
    r = rf_thresholds[driver]
    x = xgb_thresholds[driver]
    
    comparison_rows.append({
        'Variable': driver,
        'HMM Threshold': round(h['threshold'], 4),
        'HMM Range': f"[{h['threshold_lower']:.4f}, {h['threshold_upper']:.4f}]",
        'RF Threshold': round(r['threshold'], 4),
        'RF Range': f"[{r['threshold_lower']:.4f}, {r['threshold_upper']:.4f}]",
        'XGB Threshold': round(x['threshold'], 4),
        'XGB Range': f"[{x['threshold_lower']:.4f}, {x['threshold_upper']:.4f}]",
    })

threshold_comparison_df = pd.DataFrame(comparison_rows)
display(threshold_comparison_df)

# Best model recommendation
best_auc = max(hmm_auc, rf_auc, xgb_auc)
best_f1 = max(hmm_f1, rf_f1, xgb_f1)
model_names = ['HMM', 'Random Forest', 'XGBoost']
best_model_auc = model_names[[hmm_auc, rf_auc, xgb_auc].index(best_auc)]
best_model_f1 = model_names[[hmm_f1, rf_f1, xgb_f1].index(best_f1)]

print(f"\nBest by ROC-AUC: {best_model_auc} ({best_auc:.3f})")
print(f"Best by F1-Score: {best_model_f1} ({best_f1:.3f})")

MODEL PERFORMANCE COMPARISON (Monthly Data, GroupKFold CV)


,Model,Accuracy,Precision,Recall,F1_Score,ROC_AUC
0,HMM,0.72,0.500,0.036,0.067,0.511
1,Random Forest,0.75,0.600,0.321,0.419,0.640
2,XGBoost,0.80,0.667,0.571,0.615,0.663



THRESHOLD COMPARISON ACROSS ALL THREE MODELS


,Variable,HMM Threshold,HMM Range,RF Threshold,RF Range,XGB Threshold,XGB Range
0,CumCurrent,5.0426,"[5.0308, 5.0543]",3.6787,"[3.2301, 4.7335]",4.9342,"[4.9342, 4.9342]"
1,CumWaveEnergy,3450.9203,"[3046.0315, 3855.8090]",4056.0627,"[1857.0243, 9437.9842]",3224.1317,"[2241.7714, 4333.5730]"
2,Hm0_max,1.8771,"[1.8207, 1.9336]",2.4550,"[2.1175, 2.7075]",1.9300,"[1.6900, 2.1400]"
3,Hm0_mean,1.0102,"[0.9581, 1.0622]",0.7895,"[0.7397, 1.3749]",1.0152,"[0.9866, 1.1541]"
4,StormDays_wave,6.5731,"[4.5365, 8.6096]",11.7500,"[5.6250, 25.7500]",5.0000,"[5.0000, 5.0000]"
5,UcurrMax,0.3607,"[0.3606, 0.3608]",0.3297,"[0.2494, 0.3788]",0.3313,"[0.3035, 0.3376]"
6,UcurrMean,0.1646,"[0.1640, 0.1651]",0.1592,"[0.1276, 0.2010]",0.1403,"[0.1347, 0.1410]"
7,WindMax,4.3889,"[4.3161, 4.4617]",3.8650,"[3.0150, 4.9100]",3.9600,"[3.8550, 4.0650]"



Best by ROC-AUC: XGBoost (0.663)
Best by F1-Score: XGBoost (0.615)


## Section 8: Final Threshold Definition

Consolidating thresholds from all models:
- **Single-driver thresholds**: Individual driver values triggering erosion
- **Combined-driver thresholds**: Multi-driver conditions
- **Confidence levels**: Based on model agreement

In [27]:
# =============================================================================
# Section 8.1: Final Consensus Thresholds (Multi-Model)
# =============================================================================
# For each variable, compute consensus threshold ranges by combining
# the threshold estimates from HMM, Random Forest, and XGBoost.
# =============================================================================

print("="*70)
print("FINAL EROSION THRESHOLDS — MULTI-MODEL CONSENSUS")
print("="*70)

# Units and physical interpretation for key drivers
driver_meta = {
    'Hm0_max':        {'unit': 'm',     'description': 'Maximum significant wave height'},
    'Hm0_mean':       {'unit': 'm',     'description': 'Mean significant wave height'},
    'CumWaveEnergy':  {'unit': 'm²·s',  'description': 'Cumulative wave energy proxy'},
    'StormDays_wave':  {'unit': 'days',  'description': 'Storm wave days (Hm0 > 2 m)'},
    'WindMax':         {'unit': 'm/s',   'description': 'Maximum wind speed'},
    'WindMean':        {'unit': 'm/s',   'description': 'Mean wind speed'},
    'WindStressMean':  {'unit': 'N/m²',  'description': 'Mean wind stress magnitude'},
    'UcurrMax':        {'unit': 'm/s',   'description': 'Maximum ocean current speed'},
    'UcurrMean':       {'unit': 'm/s',   'description': 'Mean ocean current speed'},
    'CumCurrent':      {'unit': 'm',     'description': 'Cumulative current transport'},
}

# Model performance weights (normalised F1 scores)
f1_scores = np.array([hmm_f1, rf_f1, xgb_f1])
f1_weights = f1_scores / f1_scores.sum() if f1_scores.sum() > 0 else np.ones(3) / 3

print(f"\nModel weights (based on F1): HMM={f1_weights[0]:.3f}, RF={f1_weights[1]:.3f}, XGB={f1_weights[2]:.3f}")

# Build consensus for each variable
all_drivers = sorted(set(hmm_thresholds.keys()) & set(rf_thresholds.keys()) & set(xgb_thresholds.keys()))

final_rows = []
for driver in all_drivers:
    if driver not in driver_meta:
        continue
    
    h = hmm_thresholds[driver]
    r = rf_thresholds[driver]
    x = xgb_thresholds[driver]
    
    # Weighted consensus threshold (midpoint)
    consensus_mid = (
        f1_weights[0] * h['threshold'] +
        f1_weights[1] * r['threshold'] +
        f1_weights[2] * x['threshold']
    )
    
    # Consensus range: envelope of all three model ranges
    range_lower = min(h['threshold_lower'], r['threshold_lower'], x['threshold_lower'])
    range_upper = max(h['threshold_upper'], r['threshold_upper'], x['threshold_upper'])
    
    # Direction consensus (majority vote)
    directions = [h['direction'], r['direction'], x['direction']]
    direction = max(set(directions), key=directions.count)
    
    # Model agreement: how many models agree on direction
    agreement = sum(1 for d in directions if d == direction)
    consensus_label = 'Strong' if agreement == 3 else 'Moderate'
    
    # Feature importance rank across models (average)
    rf_imp = feature_importance[feature_importance['Feature'] == driver].index
    rf_rank = rf_imp[0] + 1 if len(rf_imp) > 0 else len(model_features)
    xgb_imp = xgb_importance[xgb_importance['Feature'] == driver].index
    xgb_rank = xgb_imp[0] + 1 if len(xgb_imp) > 0 else len(model_features)
    shap_imp = shap_importance[shap_importance['Feature'] == driver].index
    shap_rank = shap_imp[0] + 1 if len(shap_imp) > 0 else len(model_features)
    avg_rank = (rf_rank + xgb_rank + shap_rank) / 3
    
    meta = driver_meta[driver]
    
    final_rows.append({
        'Variable': driver,
        'Unit': meta['unit'],
        'Description': meta['description'],
        'Direction': direction,
        'HMM Threshold': round(h['threshold'], 4),
        'RF Threshold': round(r['threshold'], 4),
        'XGB Threshold': round(x['threshold'], 4),
        'Consensus Threshold': round(consensus_mid, 4),
        'Range Lower': round(range_lower, 4),
        'Range Upper': round(range_upper, 4),
        'Threshold Range': f"{round(range_lower, 4)} – {round(range_upper, 4)}",
        'Consensus': consensus_label,
        'Avg Importance Rank': round(avg_rank, 1),
    })

final_threshold_df = pd.DataFrame(final_rows)
final_threshold_df = final_threshold_df.sort_values('Avg Importance Rank').reset_index(drop=True)

# Display the key summary table
display_cols = ['Variable', 'Unit', 'Direction', 'HMM Threshold', 'RF Threshold',
                'XGB Threshold', 'Consensus Threshold', 'Threshold Range', 'Consensus']
print("\n")
display(final_threshold_df[display_cols])

# Compact final table with ranges
print("\n" + "="*70)
print("SUMMARY: EROSION THRESHOLD RANGES PER VARIABLE")
print("="*70)
print(f"{'Variable':<20} {'Unit':<8} {'Dir':>3}  {'Range':>30}  {'Consensus':>10}")
print("-" * 75)
for _, row in final_threshold_df.iterrows():
    print(f"{row['Variable']:<20} {row['Unit']:<8} {row['Direction']:>3}  "
          f"{row['Threshold Range']:>30}  {row['Consensus']:>10}")

print(f"\n✓ Thresholds derived from {len(y_monthly)} monthly samples across 3 models")
print(f"✓ Range = envelope across HMM, Random Forest, and XGBoost thresholds")

FINAL EROSION THRESHOLDS — MULTI-MODEL CONSENSUS

Model weights (based on F1): HMM=0.061, RF=0.380, XGB=0.559




,Variable,Unit,Direction,HMM Threshold,RF Threshold,XGB Threshold,Consensus Threshold,Threshold Range,Consensus
0,Hm0_max,m,≥,1.8771,2.4550,1.9300,2.1265,1.69 – 2.7075,Strong
1,Hm0_mean,m,≥,1.0102,0.7895,1.0152,0.9291,0.7397 – 1.3749,Moderate
2,CumWaveEnergy,m²·s,≥,3450.9203,4056.0627,3224.1317,3554.2707,1857.0243 – 9437.9842,Moderate
3,StormDays_wave,days,≥,6.5731,11.7500,5.0000,7.6625,4.5365 – 25.75,Moderate
4,WindMax,m/s,≤,4.3889,3.8650,3.9600,3.9498,3.015 – 4.91,Strong
5,UcurrMax,m/s,≥,0.3607,0.3297,0.3313,0.3325,0.2494 – 0.3788,Strong
6,UcurrMean,m/s,≤,0.1646,0.1592,0.1403,0.1489,0.1276 – 0.201,Strong
7,CumCurrent,m,≤,5.0426,3.6787,4.9342,4.4633,3.2301 – 5.0543,Moderate



SUMMARY: EROSION THRESHOLD RANGES PER VARIABLE
Variable             Unit     Dir                           Range   Consensus
---------------------------------------------------------------------------
Hm0_max              m          ≥                   1.69 – 2.7075      Strong
Hm0_mean             m          ≥                 0.7397 – 1.3749    Moderate
CumWaveEnergy        m²·s       ≥           1857.0243 – 9437.9842    Moderate
StormDays_wave       days       ≥                  4.5365 – 25.75    Moderate
WindMax              m/s        ≤                    3.015 – 4.91      Strong
UcurrMax             m/s        ≥                 0.2494 – 0.3788      Strong
UcurrMean            m/s        ≤                  0.1276 – 0.201      Strong
CumCurrent           m          ≤                 3.2301 – 5.0543    Moderate

✓ Thresholds derived from 300 monthly samples across 3 models
✓ Range = envelope across HMM, Random Forest, and XGBoost thresholds


In [28]:
# =============================================================================
# Section 8.2: Visualization of Final Consensus Thresholds
# =============================================================================

# --- A. Per-variable threshold comparison bar chart ---
fig, ax = plt.subplots(figsize=(14, 7))

vars_plot = final_threshold_df['Variable'].values
x = np.arange(len(vars_plot))
width = 0.22

hmm_vals = final_threshold_df['HMM Threshold'].values
rf_vals = final_threshold_df['RF Threshold'].values
xgb_vals = final_threshold_df['XGB Threshold'].values

ax.bar(x - width, hmm_vals, width, label='HMM', color='steelblue', alpha=0.85)
ax.bar(x, rf_vals, width, label='Random Forest', color='green', alpha=0.85)
ax.bar(x + width, xgb_vals, width, label='XGBoost', color='coral', alpha=0.85)

# Consensus range as error bars
range_lo = final_threshold_df['Range Lower'].values
range_hi = final_threshold_df['Range Upper'].values
consensus = final_threshold_df['Consensus Threshold'].values
ax.scatter(x, consensus, color='black', marker='D', s=60, zorder=5, label='Consensus')

ax.set_xticks(x)
ax.set_xticklabels(vars_plot, rotation=45, ha='right')
ax.set_ylabel('Threshold Value')
ax.set_title('Erosion Threshold Comparison Across Models')
ax.legend()
plt.tight_layout()
plt.show()

# --- B. Time-series overlay with consensus threshold bands ---
plot_drivers = ['Hm0_max', 'UcurrMax', 'WindMax', 'CumCurrent']
plot_drivers = [d for d in plot_drivers if d in final_threshold_df['Variable'].values]

n_plots = len(plot_drivers)
fig, axes = plt.subplots(n_plots, 1, figsize=(14, 4 * n_plots), sharex=True)
if n_plots == 1:
    axes = [axes]

years = env_features['monsoon_year']
colors_ts = ['red' if e == 1 else 'green' for e in env_features['Erosion_Label']]

for i, driver in enumerate(plot_drivers):
    ax = axes[i]
    row = final_threshold_df[final_threshold_df['Variable'] == driver].iloc[0]
    
    if driver in env_features.columns:
        vals = env_features[driver]
    else:
        continue
    
    ax.bar(years, vals, color=colors_ts, alpha=0.7, edgecolor='black', linewidth=0.5)
    
    # Consensus threshold band
    ax.axhspan(row['Range Lower'], row['Range Upper'], color='orange', alpha=0.2, label='Threshold range')
    ax.axhline(y=row['Consensus Threshold'], color='red', linestyle='--', linewidth=2,
               label=f"Consensus ({row['Consensus Threshold']:.3f})")
    ax.axhline(y=vals.median(), color='blue', linestyle=':', linewidth=1.5,
               label=f"Median ({vals.median():.3f})")
    
    ax.set_ylabel(f"{driver} ({row['Unit']})")
    ax.set_title(f"{driver} — Annual Values vs Threshold Range")
    ax.legend(loc='upper left', fontsize=8)
    ax.tick_params(axis='x', rotation=45)

plt.xlabel('Monsoon Year')
plt.tight_layout()
plt.show()

# --- C. Feature importance comparison across models ---
fig, ax = plt.subplots(figsize=(12, 6))

common_drivers = [d for d in final_threshold_df['Variable'].values if d in model_features][:8]

rf_norm, xgb_norm, shap_norm = [], [], []
for feat in common_drivers:
    r = feature_importance[feature_importance['Feature'] == feat]['Importance'].values
    rf_norm.append(r[0] if len(r) > 0 else 0)
    x = xgb_importance[xgb_importance['Feature'] == feat]['Importance'].values
    xgb_norm.append(x[0] if len(x) > 0 else 0)
    s = shap_importance[shap_importance['Feature'] == feat]['Mean_SHAP'].values
    shap_norm.append(s[0] if len(s) > 0 else 0)

rf_norm = np.array(rf_norm) / (max(rf_norm) if max(rf_norm) > 0 else 1)
xgb_norm = np.array(xgb_norm) / (max(xgb_norm) if max(xgb_norm) > 0 else 1)
shap_norm = np.array(shap_norm) / (max(shap_norm) if max(shap_norm) > 0 else 1)

x_pos = np.arange(len(common_drivers))
w = 0.25
ax.bar(x_pos - w, rf_norm, w, label='Random Forest', color='steelblue')
ax.bar(x_pos, xgb_norm, w, label='XGBoost (Gain)', color='coral')
ax.bar(x_pos + w, shap_norm, w, label='SHAP', color='green')
ax.set_xticks(x_pos)
ax.set_xticklabels(common_drivers, rotation=45, ha='right')
ax.set_ylabel('Normalised Importance')
ax.set_title('Feature Importance Comparison Across Models')
ax.legend()
plt.tight_layout()
plt.show()

In [29]:
# =============================================================================
# Section 8.3: Export Results
# =============================================================================

# Save processed environmental features
env_features.to_csv(r'D:\Kanjana\Coastal_Research_GitHub\coastalai\backend\uploads/processed_annual_features.csv', index=False)

# Build export data from the computed consensus thresholds
threshold_export = []
for _, row in final_threshold_df.iterrows():
    driver = row['Variable']
    threshold_export.append({
        'Driver': driver,
        'Unit': row['Unit'],
        'Description': row['Description'],
        'Direction': row['Direction'],
        'HMM_Threshold': float(row['HMM Threshold']),
        'RF_Threshold': float(row['RF Threshold']),
        'XGB_Threshold': float(row['XGB Threshold']),
        'Consensus_Threshold': float(row['Consensus Threshold']),
        'Erosion_Threshold_Lower': float(row['Range Lower']),
        'Erosion_Threshold_Upper': float(row['Range Upper']),
        'Model_Consensus': row['Consensus'],
    })

threshold_export_df = pd.DataFrame(threshold_export)
threshold_export_df.to_csv(r'D:\Kanjana\Coastal_Research_GitHub\coastalai\backend\uploads/erosion_thresholds.csv', index=False)
threshold_export_df.to_json(r'D:\Kanjana\Coastal_Research_GitHub\coastalai\backend\uploads/erosion_thresholds.json', orient='records', indent=2)

# Save model results summary
model_results = {
    'shoreline_stats': transect_stats,
    'hmm_thresholds': {k: {kk: float(vv) if isinstance(vv, (np.floating, float)) else vv 
                           for kk, vv in v.items()} for k, v in hmm_thresholds.items()},
    'rf_feature_importance': feature_importance.to_dict(orient='records'),
    'xgb_feature_importance': xgb_importance.to_dict(orient='records'),
    'shap_importance': shap_importance.to_dict(orient='records'),
    'model_comparison': model_comparison.to_dict(orient='records'),
}

print("="*60)
print("RESULTS EXPORTED")
print("="*60)
print(f"✓ Annual features: {DATA_PATH}/processed_annual_features.csv")
print(f"✓ Threshold summary: {DATA_PATH}/erosion_thresholds.csv")
print(f"✓ Threshold summary JSON: {DATA_PATH}/erosion_thresholds.json")
print("\nFinal threshold summary:")
display(threshold_export_df)

RESULTS EXPORTED
✓ Annual features: D:\Kanjana\Coastal_Research_GitHub\coastalai\backend\uploads/processed_annual_features.csv
✓ Threshold summary: D:\Kanjana\Coastal_Research_GitHub\coastalai\backend\uploads/erosion_thresholds.csv
✓ Threshold summary JSON: D:\Kanjana\Coastal_Research_GitHub\coastalai\backend\uploads/erosion_thresholds.json

Final threshold summary:


,Driver,Unit,Description,Direction,HMM_Threshold,RF_Threshold,XGB_Threshold,Consensus_Threshold,Erosion_Threshold_Lower,Erosion_Threshold_Upper,Model_Consensus
0,Hm0_max,m,Maximum significant wave height,≥,1.8771,2.4550,1.9300,2.1265,1.6900,2.7075,Strong
1,Hm0_mean,m,Mean significant wave height,≥,1.0102,0.7895,1.0152,0.9291,0.7397,1.3749,Moderate
2,CumWaveEnergy,m²·s,Cumulative wave energy proxy,≥,3450.9203,4056.0627,3224.1317,3554.2707,1857.0243,9437.9842,Moderate
3,StormDays_wave,days,Storm wave days (Hm0 > 2 m),≥,6.5731,11.7500,5.0000,7.6625,4.5365,25.7500,Moderate
4,WindMax,m/s,Maximum wind speed,≤,4.3889,3.8650,3.9600,3.9498,3.0150,4.9100,Strong
5,UcurrMax,m/s,Maximum ocean current speed,≥,0.3607,0.3297,0.3313,0.3325,0.2494,0.3788,Strong
6,UcurrMean,m/s,Mean ocean current speed,≤,0.1646,0.1592,0.1403,0.1489,0.1276,0.2010,Strong
7,CumCurrent,m,Cumulative current transport,≤,5.0426,3.6787,4.9342,4.4633,3.2301,5.0543,Moderate


In [30]:
# =============================================================================
# Section 8.4: Export All Data for Frontend Web Application
# =============================================================================

import json
import shutil
import os

# Create frontend data directory - use the actual frontend path
# Get the notebook's directory and build path to frontend
notebook_dir = os.path.dirname(os.path.abspath('__file__'))
frontend_data_path = os.path.join(notebook_dir, 'frontend', 'public', 'data')
os.makedirs(frontend_data_path, exist_ok=True)

print("="*60)
print(f"Frontend data path: {frontend_data_path}")
print("="*60)

# Copy erosion_thresholds.json to frontend data directory
source_json = f'{DATA_PATH}/erosion_thresholds.json'
dest_json = os.path.join(frontend_data_path, 'erosion_thresholds.json')

if os.path.exists(source_json):
    shutil.copy(source_json, dest_json)
    print(f"✓ Copied threshold JSON to frontend: {dest_json}")
else:
    print(f"⚠ Warning: Source file not found: {source_json}")

# 1. Shoreline Data (use 'id' column which exists in the CSV)
shoreline_export = shoreline_df[['id', 'NSM', 'EPR', 'SCE', 'LRR', 'Erosion_Label', 'Erosion_Binary']].copy()
shoreline_export = shoreline_export.rename(columns={'id': 'TransectId'})
shoreline_export = shoreline_export.to_dict(orient='records')

# 2. Time Series Data (Environmental Features by Year) - use env_features_monthly which has HMM_State
# Get available columns from env_features_monthly
available_export_cols = [col for col in ['monsoon_year', 'Hm0_max', 'Hm0_mean', 'CumWaveEnergy', 'StormDays_wave',
    'WindMax', 'WindMean', 'WindStressMean', 'UcurrMax', 'UcurrMean', 
    'CumCurrent', 'Erosion_Label', 'HMM_State', 'Forcing_Regime'] if col in env_features_monthly.columns]
timeseries_export = env_features_monthly[available_export_cols].to_dict(orient='records')

# 3. HMM State Distribution  
state_counts = env_features_monthly['HMM_State'].value_counts()
hmm_state_distribution = []
for state in sorted(state_counts.index):
    label = 'Normal' if state == 0 else ('High Risk' if state == 1 else 'Extreme')
    color = '#3b82f6' if state == 0 else ('#ef4444' if state == 1 else '#dc2626')
    hmm_state_distribution.append({
        'state': label, 
        'count': int(state_counts.get(state, 0)), 
        'percentage': round(state_counts.get(state, 0) / len(env_features_monthly) * 100, 1), 
        'color': color
    })

# 4. HMM State Means
normal_states_list = [0]  # State 0 is normal
risk_states_list = [s for s in state_counts.index if s > 0]  # Other states are risk
normal_means = env_features_monthly[env_features_monthly['HMM_State'].isin(normal_states_list)][['Hm0_max', 'UcurrMax', 'WindMax']].mean()
high_risk_means = env_features_monthly[env_features_monthly['HMM_State'].isin(risk_states_list)][['Hm0_max', 'UcurrMax', 'WindMax']].mean()
hmm_state_means = {
    'normal': {'Hm0_max': round(float(normal_means['Hm0_max']), 2),
               'UcurrMax': round(float(normal_means['UcurrMax']), 3),
               'WindMax': round(float(normal_means['WindMax']), 2)},
    'highRisk': {'Hm0_max': round(float(high_risk_means['Hm0_max']), 2),
                 'UcurrMax': round(float(high_risk_means['UcurrMax']), 3),
                 'WindMax': round(float(high_risk_means['WindMax']), 2)}
}

# 5. HMM Probability Data (state probabilities over time)
hmm_prob_data = []

Frontend data path: D:\Kanjana\Coastal_Research_GitHub\coastalai\backend\uploads\frontend\public\data
✓ Copied threshold JSON to frontend: D:\Kanjana\Coastal_Research_GitHub\coastalai\backend\uploads\frontend\public\data\erosion_thresholds.json


## Section 10: Meteorological Threshold Forecasting (SARIMA)

Using all historical monthly environmental data to learn seasonal patterns and trends
for each key variable, then forecast threshold values for the next 6, 12, 18, and 24 months.

**Approach:**
- Train individual SARIMA (Seasonal ARIMA) models on each of the 5 key variables
- SARIMA captures both non-seasonal (trend) and seasonal (monsoon cycle) patterns
- Automatic order selection using AIC minimisation with grid search
- Generate point forecasts with 95% confidence intervals
- Compare forecasted values against computed erosion thresholds
- Predict exceedance probabilities for each horizon

In [31]:
# =============================================================================
# Section 10.1: SARIMA Model Training & Forecasting
# =============================================================================
# Uses historical monthly time series for each key variable to forecast
# future threshold values at 6, 12, 18, and 24 month horizons.
# =============================================================================

import itertools
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller

# ── Target variables (the 5 critical erosion drivers) ──
FORECAST_VARS = ['Hm0_max', 'UcurrMax', 'WindMax', 'CumCurrent', 'StormDays_wave']
FORECAST_HORIZONS = [6, 12, 18, 24]  # months

# ── Prepare monthly time series ──
# Sort by time and create proper datetime index
ts_data = env_features_monthly[['year_month'] + FORECAST_VARS].copy()
ts_data = ts_data.sort_values('year_month').reset_index(drop=True)
ts_data = ts_data.set_index('year_month')

# Fill any remaining gaps via interpolation
ts_data = ts_data.interpolate(method='time').ffill().bfill()

# Ensure frequency is monthly
ts_data = ts_data.asfreq('M')
if ts_data.isnull().sum().sum() > 0:
    ts_data = ts_data.interpolate(method='time').ffill().bfill()

print("="*70)
print("SARIMA FORECASTING – MONTHLY ENVIRONMENTAL DATA")
print("="*70)
print(f"Time range: {ts_data.index.min()} → {ts_data.index.max()}")
print(f"Total months: {len(ts_data)}")
print(f"Variables: {FORECAST_VARS}")
print(f"Horizons: {FORECAST_HORIZONS} months")

# ── Stationarity test ──
print(f"\n{'─'*60}")
print("ADF Stationarity Tests (p < 0.05 → stationary)")
print(f"{'─'*60}")
for var in FORECAST_VARS:
    result = adfuller(ts_data[var].dropna(), autolag='AIC')
    status = "✓ Stationary" if result[1] < 0.05 else "✗ Non-stationary (needs differencing)"
    print(f"  {var:20s}: p={result[1]:.4f}  {status}")

# =============================================================================
# SARIMA Grid-Search with AIC selection
# =============================================================================

def fit_best_sarima(series, seasonal_period=12, max_order=2):
    """
    Grid-search SARIMA orders and return the best model by AIC.
    Uses a constrained grid (p,d,q ≤ max_order) to keep runtime reasonable.
    """
    best_aic = np.inf
    best_model = None
    best_order = None
    best_seasonal = None

    # Determine differencing order from ADF test
    adf_p = adfuller(series.dropna(), autolag='AIC')[1]
    d = 0 if adf_p < 0.05 else 1
    D = 1  # seasonal differencing (monsoon cycle)

    # Grid-search p, q (non-seasonal) and P, Q (seasonal)
    p_range = range(0, max_order + 1)
    q_range = range(0, max_order + 1)
    P_range = range(0, 2)
    Q_range = range(0, 2)

    for p, q, P, Q in itertools.product(p_range, q_range, P_range, Q_range):
        try:
            model = SARIMAX(
                series,
                order=(p, d, q),
                seasonal_order=(P, D, Q, seasonal_period),
                enforce_stationarity=False,
                enforce_invertibility=False,
            )
            res = model.fit(disp=False, maxiter=200)
            if res.aic < best_aic:
                best_aic = res.aic
                best_model = res
                best_order = (p, d, q)
                best_seasonal = (P, D, Q, seasonal_period)
        except Exception:
            continue

    return best_model, best_order, best_seasonal, best_aic

# ── Train models and generate forecasts ──
forecast_results = {}
sarima_models = {}
sarima_diagnostics = {}

print(f"\n{'='*70}")
print("TRAINING SARIMA MODELS (this may take a minute...)")
print(f"{'='*70}")

for var in FORECAST_VARS:
    print(f"\n{'─'*60}")
    print(f"Training: {var}")
    print(f"{'─'*60}")

    series = ts_data[var].dropna()

    # Fit best SARIMA
    best_model, best_order, best_seasonal, best_aic = fit_best_sarima(series)

    if best_model is None:
        print(f"  ⚠ Failed to fit SARIMA for {var}")
        continue

    sarima_models[var] = best_model
    print(f"  ✓ Best order: SARIMA{best_order}x{best_seasonal}")
    print(f"  ✓ AIC: {best_aic:.1f}")

    # In-sample diagnostics
    resid = best_model.resid
    sarima_diagnostics[var] = {
        'order': best_order,
        'seasonal_order': best_seasonal,
        'aic': best_aic,
        'bic': best_model.bic,
        'mae': np.mean(np.abs(resid)),
        'rmse': np.sqrt(np.mean(resid**2)),
        'mape': np.mean(np.abs(resid / series.iloc[-len(resid):])) * 100 if len(series) >= len(resid) else 0,
    }
    print(f"  ✓ MAE: {sarima_diagnostics[var]['mae']:.4f}")
    print(f"  ✓ RMSE: {sarima_diagnostics[var]['rmse']:.4f}")

    # Forecast each horizon
    max_horizon = max(FORECAST_HORIZONS)
    fc = best_model.get_forecast(steps=max_horizon)
    fc_mean = fc.predicted_mean
    fc_ci = fc.conf_int(alpha=0.05)  # 95% CI

    forecast_results[var] = {
        'mean': fc_mean,
        'ci_lower': fc_ci.iloc[:, 0],
        'ci_upper': fc_ci.iloc[:, 1],
    }

    for h in FORECAST_HORIZONS:
        avg = fc_mean.iloc[:h].mean()
        peak = fc_mean.iloc[:h].max()
        print(f"  → {h:2d}-month avg: {avg:.4f}  |  peak: {peak:.4f}")

print(f"\n✓ All {len(sarima_models)} SARIMA models trained successfully")

SARIMA FORECASTING – MONTHLY ENVIRONMENTAL DATA
Time range: 2000-04 → 2025-03
Total months: 300
Variables: ['Hm0_max', 'UcurrMax', 'WindMax', 'CumCurrent', 'StormDays_wave']
Horizons: [6, 12, 18, 24] months

────────────────────────────────────────────────────────────
ADF Stationarity Tests (p < 0.05 → stationary)
────────────────────────────────────────────────────────────
  Hm0_max             : p=0.0039  ✓ Stationary
  UcurrMax            : p=0.0003  ✓ Stationary
  WindMax             : p=0.0149  ✓ Stationary
  CumCurrent          : p=0.0001  ✓ Stationary
  StormDays_wave      : p=0.0087  ✓ Stationary

TRAINING SARIMA MODELS (this may take a minute...)

────────────────────────────────────────────────────────────
Training: Hm0_max
────────────────────────────────────────────────────────────


  ✓ Best order: SARIMA(1, 0, 0)x(0, 1, 1, 12)
  ✓ AIC: 205.5
  ✓ MAE: 0.3233
  ✓ RMSE: 0.5108
  →  6-month avg: 2.2669  |  peak: 2.6035
  → 12-month avg: 1.8572  |  peak: 2.6035
  → 18-month avg: 1.9934  |  peak: 2.6035
  → 24-month avg: 1.8570  |  peak: 2.6035

────────────────────────────────────────────────────────────
Training: UcurrMax
────────────────────────────────────────────────────────────


  ✓ Best order: SARIMA(0, 0, 0)x(1, 1, 1, 12)
  ✓ AIC: -678.4
  ✓ MAE: 0.0634
  ✓ RMSE: 0.0947
  →  6-month avg: 0.2957  |  peak: 0.3356
  → 12-month avg: 0.3153  |  peak: 0.4266
  → 18-month avg: 0.3087  |  peak: 0.4266
  → 24-month avg: 0.3152  |  peak: 0.4639

────────────────────────────────────────────────────────────
Training: WindMax
────────────────────────────────────────────────────────────


D:\Kanjana\Coastal_Research\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


  ✓ Best order: SARIMA(2, 0, 0)x(0, 1, 1, 12)
  ✓ AIC: 481.9
  ✓ MAE: 0.5951
  ✓ RMSE: 0.9402
  →  6-month avg: 5.4035  |  peak: 6.2102
  → 12-month avg: 4.4629  |  peak: 6.2102
  → 18-month avg: 4.7921  |  peak: 6.2353
  → 24-month avg: 4.4751  |  peak: 6.2353

────────────────────────────────────────────────────────────
Training: CumCurrent
────────────────────────────────────────────────────────────


D:\Kanjana\Coastal_Research\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


  ✓ Best order: SARIMA(0, 0, 2)x(0, 1, 1, 12)
  ✓ AIC: 760.6
  ✓ MAE: 0.9133
  ✓ RMSE: 1.3840
  →  6-month avg: 4.4826  |  peak: 5.8465
  → 12-month avg: 4.4761  |  peak: 5.8465
  → 18-month avg: 4.4783  |  peak: 5.8465
  → 24-month avg: 4.4761  |  peak: 5.8465

────────────────────────────────────────────────────────────
Training: StormDays_wave
────────────────────────────────────────────────────────────


  ✓ Best order: SARIMA(1, 0, 2)x(0, 1, 1, 12)
  ✓ AIC: 2287.4
  ✓ MAE: 9.2468
  ✓ RMSE: 17.0240
  →  6-month avg: 25.5163  |  peak: 48.7825
  → 12-month avg: 13.7444  |  peak: 48.7825
  → 18-month avg: 17.6676  |  peak: 48.7825
  → 24-month avg: 13.7438  |  peak: 48.7825

✓ All 5 SARIMA models trained successfully


In [32]:
# =============================================================================
# Section 10.2: Threshold Exceedance Analysis & Forecast Export
# =============================================================================
# Compare forecasted values against HMM-derived thresholds.
# Calculate exceedance probability at each horizon using the confidence interval.
# =============================================================================

from scipy import stats

print("="*70)
print("THRESHOLD EXCEEDANCE ANALYSIS")
print("="*70)

# ── Build structured forecast data for export ──
forecast_export = {}

for var in FORECAST_VARS:
    if var not in forecast_results:
        continue
    
    fc = forecast_results[var]
    model = sarima_models[var]
    diag = sarima_diagnostics[var]
    
    # Get threshold (from HMM thresholds computed earlier)
    threshold_val = hmm_thresholds[var]['threshold'] if var in hmm_thresholds else None
    threshold_dir = hmm_thresholds[var]['direction'] if var in hmm_thresholds else '≥'
    
    # Historical statistics for context
    hist_series = ts_data[var].dropna()
    hist_mean = float(hist_series.mean())
    hist_std = float(hist_series.std())
    hist_min = float(hist_series.min())
    hist_max = float(hist_series.max())
    
    var_forecasts = {}
    
    for h in FORECAST_HORIZONS:
        # Slice forecast to horizon
        fc_mean_h = fc['mean'].iloc[:h]
        fc_lower_h = fc['ci_lower'].iloc[:h]
        fc_upper_h = fc['ci_upper'].iloc[:h]
        
        # Monthly forecast data (for charts)
        monthly_data = []
        for i in range(h):
            month_date = fc_mean_h.index[i]
            monthly_data.append({
                'date': month_date.strftime('%Y-%m'),
                'month': int(month_date.month),
                'predicted': round(float(fc_mean_h.iloc[i]), 4),
                'ci_lower': round(float(fc_lower_h.iloc[i]), 4),
                'ci_upper': round(float(fc_upper_h.iloc[i]), 4),
            })
        
        # Aggregate statistics for this horizon
        avg_forecast = float(fc_mean_h.mean())
        peak_forecast = float(fc_mean_h.max())
        min_forecast = float(fc_mean_h.min())
        
        # Exceedance probability (using forecast distribution)
        # The forecast standard error grows over time; we use the CI width
        # to estimate the forecast distribution at each step
        exceedance_months = 0
        if threshold_val is not None:
            for i in range(h):
                pred = fc_mean_h.iloc[i]
                ci_width = fc_upper_h.iloc[i] - fc_lower_h.iloc[i]
                se = ci_width / (2 * 1.96)  # approximate std error from 95% CI
                if se > 0:
                    if threshold_dir == '≥':
                        # P(X >= threshold) = 1 - Phi((threshold - pred) / se)
                        z = (threshold_val - pred) / se
                        p_exceed = 1 - stats.norm.cdf(z)
                    else:
                        z = (pred - threshold_val) / se
                        p_exceed = 1 - stats.norm.cdf(z)
                else:
                    p_exceed = 1.0 if pred >= threshold_val else 0.0
                if p_exceed > 0.5:
                    exceedance_months += 1
        
        exceedance_pct = (exceedance_months / h * 100) if h > 0 else 0
        
        # Trend: compare average forecast to historical mean
        trend_pct = ((avg_forecast - hist_mean) / hist_mean * 100) if hist_mean != 0 else 0
        
        var_forecasts[str(h)] = {
            'monthly': monthly_data,
            'avgForecast': round(avg_forecast, 4),
            'peakForecast': round(peak_forecast, 4),
            'minForecast': round(min_forecast, 4),
            'exceedanceMonths': exceedance_months,
            'exceedancePct': round(exceedance_pct, 1),
            'trendPct': round(trend_pct, 1),
        }
    
    # Risk level based on 12-month exceedance
    exc_12 = var_forecasts.get('12', {}).get('exceedancePct', 0)
    if exc_12 >= 50:
        risk_level = 'High'
    elif exc_12 >= 25:
        risk_level = 'Medium'
    else:
        risk_level = 'Low'
    
    forecast_export[var] = {
        'horizons': var_forecasts,
        'threshold': round(threshold_val, 4) if threshold_val is not None else None,
        'thresholdDirection': threshold_dir,
        'historicalMean': round(hist_mean, 4),
        'historicalStd': round(hist_std, 4),
        'historicalMin': round(hist_min, 4),
        'historicalMax': round(hist_max, 4),
        'riskLevel': risk_level,
        'model': {
            'order': list(diag['order']),
            'seasonalOrder': list(diag['seasonal_order']),
            'aic': round(diag['aic'], 1),
            'bic': round(diag['bic'], 1),
            'mae': round(diag['mae'], 4),
            'rmse': round(diag['rmse'], 4),
        }
    }
    
    print(f"\n{'─'*60}")
    print(f"{var}: Threshold = {threshold_val:.4f} ({threshold_dir})")
    print(f"  Historical: mean={hist_mean:.4f}, std={hist_std:.4f}")
    for h in FORECAST_HORIZONS:
        vf = var_forecasts[str(h)]
        print(f"  {h:2d}-month: avg={vf['avgForecast']:.4f}, peak={vf['peakForecast']:.4f}, "
              f"exceedance={vf['exceedancePct']:.0f}%, trend={vf['trendPct']:+.1f}%")
    print(f"  → Risk Level: {risk_level}")

# ── Cross-validation (last 24 months held out) ──
print(f"\n{'='*70}")
print("MODEL VALIDATION (24-month hold-out)")
print(f"{'='*70}")

cv_results = {}
for var in FORECAST_VARS:
    if var not in sarima_models:
        continue
    
    series = ts_data[var].dropna()
    n = len(series)
    holdout = min(24, n // 4)  # Use 24 months or 25% whichever is smaller
    train = series.iloc[:-holdout]
    test = series.iloc[-holdout:]
    
    # Refit on training data with same orders
    diag = sarima_diagnostics[var]
    try:
        cv_model = SARIMAX(
            train,
            order=diag['order'],
            seasonal_order=diag['seasonal_order'],
            enforce_stationarity=False,
            enforce_invertibility=False,
        ).fit(disp=False, maxiter=200)
        
        cv_fc = cv_model.get_forecast(steps=holdout)
        cv_pred = cv_fc.predicted_mean
        
        # Align indices
        cv_actual = test.values
        cv_predicted = cv_pred.values[:len(cv_actual)]
        
        mae = np.mean(np.abs(cv_actual - cv_predicted))
        rmse = np.sqrt(np.mean((cv_actual - cv_predicted)**2))
        mape = np.mean(np.abs((cv_actual - cv_predicted) / np.where(cv_actual == 0, 1, cv_actual))) * 100
        
        # Correlation
        corr = np.corrcoef(cv_actual, cv_predicted)[0, 1]
        
        cv_results[var] = {
            'mae': round(float(mae), 4),
            'rmse': round(float(rmse), 4),
            'mape': round(float(mape), 1),
            'correlation': round(float(corr), 4),
            'holdoutMonths': holdout,
        }
        
        print(f"  {var:20s}: MAE={mae:.4f}  RMSE={rmse:.4f}  MAPE={mape:.1f}%  r={corr:.3f}")
        
        # Add to export
        forecast_export[var]['validation'] = cv_results[var]
        
    except Exception as e:
        print(f"  {var:20s}: CV failed – {e}")

# ── Overall risk assessment ──
print(f"\n{'='*70}")
print("COMPOSITE RISK FORECAST")
print(f"{'='*70}")

risk_counts = {'High': 0, 'Medium': 0, 'Low': 0}
for var in forecast_export:
    risk_counts[forecast_export[var]['riskLevel']] += 1

overall_risk = 'High' if risk_counts['High'] >= 2 else ('Medium' if risk_counts['High'] >= 1 or risk_counts['Medium'] >= 2 else 'Low')
print(f"  Variables at High risk:   {risk_counts['High']}")
print(f"  Variables at Medium risk: {risk_counts['Medium']}")
print(f"  Variables at Low risk:    {risk_counts['Low']}")
print(f"  ═══ Overall Risk: {overall_risk} ═══")

# Store for export — use + 1 for PeriodIndex (not DateOffset)
_last_period = ts_data.index.max()
_forecast_start = _last_period + 1

forecast_metadata = {
    'forecastHorizons': FORECAST_HORIZONS,
    'variables': FORECAST_VARS,
    'overallRisk': overall_risk,
    'dataRange': f"{ts_data.index.min().strftime('%Y-%m')} to {_last_period.strftime('%Y-%m')}",
    'totalMonths': len(ts_data),
    'forecastFrom': _forecast_start.strftime('%Y-%m'),
}

print(f"\n✓ Forecast export ready: {len(forecast_export)} variables × {len(FORECAST_HORIZONS)} horizons")


THRESHOLD EXCEEDANCE ANALYSIS

────────────────────────────────────────────────────────────
Hm0_max: Threshold = 1.8771 (≥)
  Historical: mean=1.8332, std=0.6203
   6-month: avg=2.2669, peak=2.6035, exceedance=83%, trend=+23.7%
  12-month: avg=1.8572, peak=2.6035, exceedance=50%, trend=+1.3%
  18-month: avg=1.9934, peak=2.6035, exceedance=61%, trend=+8.7%
  24-month: avg=1.8570, peak=2.6035, exceedance=50%, trend=+1.3%
  → Risk Level: High

────────────────────────────────────────────────────────────
UcurrMax: Threshold = 0.3607 (≥)
  Historical: mean=0.3112, std=0.0794
   6-month: avg=0.2957, peak=0.3356, exceedance=0%, trend=-5.0%
  12-month: avg=0.3153, peak=0.4266, exceedance=17%, trend=+1.3%
  18-month: avg=0.3087, peak=0.4266, exceedance=11%, trend=-0.8%
  24-month: avg=0.3152, peak=0.4639, exceedance=17%, trend=+1.3%
  → Risk Level: Low

────────────────────────────────────────────────────────────
WindMax: Threshold = 4.3889 (≤)
  Historical: mean=4.6215, std=1.2961
   6-month: 

  Hm0_max             : MAE=0.3198  RMSE=0.4086  MAPE=17.9%  r=0.782


  UcurrMax            : MAE=0.0618  RMSE=0.0833  MAPE=22.4%  r=0.497
  WindMax             : MAE=0.4371  RMSE=0.5539  MAPE=12.1%  r=0.938


  CumCurrent          : MAE=0.8133  RMSE=1.1595  MAPE=19.4%  r=0.552


  StormDays_wave      : MAE=13.5261  RMSE=22.2342  MAPE=463.2%  r=0.505

COMPOSITE RISK FORECAST
  Variables at High risk:   3
  Variables at Medium risk: 0
  Variables at Low risk:    2
  ═══ Overall Risk: High ═══

✓ Forecast export ready: 5 variables × 4 horizons


In [33]:
# =============================================================================
# Section 10.3: Forecast Visualisation
# =============================================================================

fig, axes = plt.subplots(len(FORECAST_VARS), 1, figsize=(14, 4*len(FORECAST_VARS)), sharex=False)
if len(FORECAST_VARS) == 1:
    axes = [axes]

for idx, var in enumerate(FORECAST_VARS):
    ax = axes[idx]
    series = ts_data[var].dropna()
    
    # Convert PeriodIndex to timestamps for matplotlib compatibility
    hist_tail = series.iloc[-48:]
    hist_dates = hist_tail.index.to_timestamp() if hasattr(hist_tail.index, 'to_timestamp') else hist_tail.index
    ax.plot(hist_dates, hist_tail.values, 'b-', linewidth=1.2, label='Historical')
    
    if var in forecast_results:
        fc = forecast_results[var]
        fc_mean = fc['mean']
        fc_lower = fc['ci_lower']
        fc_upper = fc['ci_upper']
        
        fc_dates = fc_mean.index.to_timestamp() if hasattr(fc_mean.index, 'to_timestamp') else fc_mean.index
        ax.plot(fc_dates, fc_mean.values, 'r-', linewidth=1.5, label='Forecast')
        ax.fill_between(fc_dates, fc_lower.values, fc_upper.values, 
                        color='red', alpha=0.15, label='95% CI')
    
    # Threshold line
    if var in hmm_thresholds:
        thresh = hmm_thresholds[var]['threshold']
        ax.axhline(y=thresh, color='orange', linestyle='--', linewidth=1.5, 
                   label=f'Threshold ({thresh:.3f})')
    
    ax.set_title(f'{var}', fontsize=13, fontweight='bold')
    ax.set_ylabel(var)
    ax.legend(loc='upper left', fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('SARIMA Forecasts with Erosion Thresholds', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{DATA_PATH}/forecast_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Forecast plots saved")


✓ Forecast plots saved


In [34]:

# =============================================================================
# AUTO-GENERATED: Comprehensive JSON export for frontend
# =============================================================================
import json, os, base64, io, numpy as np

_RESULTS_PATH = r"D:\Kanjana\Coastal_Research_GitHub\coastalai\backend\results\results_96db02d305d2.json"
_FRONTEND_PATH = r"D:\Kanjana\Coastal_Research_GitHub\coastalai\frontend\public\data"

def _safe(v):
    """Make a value JSON-serialisable."""
    if isinstance(v, (np.integer,)):
        return int(v)
    if isinstance(v, (np.floating,)):
        return float(v)
    if isinstance(v, np.ndarray):
        return v.tolist()
    if hasattr(v, 'item'):
        return v.item()
    return v

def _fig_to_b64(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=100, bbox_inches="tight")
    buf.seek(0)
    return base64.b64encode(buf.read()).decode()

# ---- Build the results dict ----
results = {}

# 1. Summary
results["summary"] = {
    "totalTransects": _safe(transect_stats.get("Total_Transects", 0)),
    "erodingTransects": _safe(int(transect_stats.get("Pct_Eroding", 0) / 100 * transect_stats.get("Total_Transects", 0))),
    "erosionRate": _safe(round(transect_stats.get("Pct_Eroding", 0), 1)),
    "meanNSM": _safe(round(transect_stats.get("Mean_NSM", 0), 2)),
    "medianNSM": _safe(round(transect_stats.get("Median_NSM", 0), 2)),
    "totalYears": _safe(len(shoreline_annual)),
    "erosionYears": _safe(int(shoreline_annual["Erosion_Binary"].sum())),
    "analysisYearRange": f"{int(shoreline_annual['year'].min())}-{int(shoreline_annual['year'].max())}",
    "beachState": beach_state,
    "meanEPR": _safe(round(transect_stats.get("Mean_EPR", 0), 2)),
    "meanLRR": _safe(round(transect_stats.get("Mean_LRR", 0), 2)),
}

# 2. Yearly shoreline
results["yearlyShoreline"] = yearly_shoreline_export

# 3. Shoreline transects
results["shoreline"] = shoreline_export

# 4. Time series (monthly env + annual erosion labels)
_ts_cols = [c for c in ['monsoon_year', 'Hm0_max', 'Hm0_mean', 'CumWaveEnergy', 'StormDays_wave',
                         'WindMax', 'WindMean', 'WindStressMean', 'UcurrMax', 'UcurrMean',
                         'CumCurrent', 'Erosion_Label', 'HMM_State'] if c in env_features_monthly.columns]
results["timeSeries"] = env_features_monthly[_ts_cols].to_dict(orient='records')

# 5. Scatter data (annual: wave height vs NSM)
results["scatter"] = [
    {"Hm0_max": _safe(row["Hm0_max"]), "annual_NSM": _safe(row["annual_NSM"]),
      "Erosion_Label": _safe(row["Erosion_Label"]), "year": _safe(int(row["monsoon_year"]))}
    for _, row in env_features.iterrows()
]

# 6. Correlation matrix
try:
    _corr_feats = [c for c in ['Hm0_max', 'Hm0_mean', 'CumWaveEnergy', 'StormDays_wave',
                                'WindMax', 'WindMean', 'WindStressMean',
                                'UcurrMax', 'UcurrMean', 'CumCurrent', 'Erosion_Label']
                   if c in env_features.columns]
    _cm = env_features[_corr_feats].corr()
    results["correlation"] = {
        "features": _corr_feats,
        "matrix": _cm.values.tolist(),
    }
except Exception:
    results["correlation"] = None

# 7. PCA
try:
    results["pca"] = {
        "data": [{"PC1": _safe(r["PC1"]), "PC2": _safe(r["PC2"]),
                    "PC3": _safe(r.get("PC3", 0)),
                    "year": _safe(int(r["monsoon_year"])),
                    "Erosion_Label": _safe(r["Erosion_Label"])}
                  for _, r in env_features.iterrows()],
        "variance": [_safe(v) for v in pca.explained_variance_ratio_],
        "loadings": pca.components_.tolist(),
        "features": pca_features,
    }
except Exception:
    results["pca"] = None

# 8. Forcing regimes
results["forcingRegimes"] = forcing_regimes_export

# 9. Boxplot data
results["boxplot"] = boxplot_export

# 10. ROC data
try:
    results["roc"] = {
        "hmm": {"fpr": fpr_hmm.tolist(), "tpr": tpr_hmm.tolist(), "auc": _safe(hmm_auc)},
        "rf":  {"fpr": fpr_rf.tolist(),  "tpr": tpr_rf.tolist(),  "auc": _safe(rf_auc)},
        "xgb": {"fpr": fpr_xgb.tolist(), "tpr": tpr_xgb.tolist(), "auc": _safe(xgb_auc)},
    }
except Exception:
    results["roc"] = None

# 11. Models
# -- Random Forest --
try:
    # Export ALL thresholds (sorted by importance), not just top 5
    _importance_order = feature_importance["Feature"].tolist()
    _rf_thresh = {}
    for feat in _importance_order:
        if feat in rf_thresholds:
            _rf_thresh[feat] = {
                "value": _safe(round(rf_thresholds[feat]["threshold"], 3)),
                "lower": _safe(round(rf_thresholds[feat]["threshold_lower"], 3)),
                "upper": _safe(round(rf_thresholds[feat]["threshold_upper"], 3)),
                "nSplits": _safe(rf_thresholds[feat]["n_splits"]),
                "condition": rf_thresholds[feat].get("direction", "≥"),
                "unit": "m" if "Hm0" in feat else "m/s" if "curr" in feat.lower() or "Wind" in feat else "",
            }

    # Model configuration from best estimator
    _bp = rf_search.best_params_ if hasattr(rf_search, 'best_params_') else {}
    _rf_config = {
        "nEstimators": _safe(rf_model.n_estimators),
        "maxDepth": _safe(_bp.get("max_depth", None)),
        "minSamplesSplit": _safe(_bp.get("min_samples_split", 2)),
        "minSamplesLeaf": _safe(_bp.get("min_samples_leaf", 1)),
        "maxFeatures": _safe(str(_bp.get("max_features", "sqrt"))),
        "criterion": _safe(getattr(rf_model, "criterion", "gini")),
        "classWeight": _safe(str(_bp.get("class_weight", "balanced"))),
        "bootstrap": _safe(getattr(rf_model, "bootstrap", True)),
        "cvFolds": _safe(int(n_splits)),
        "randomState": 42,
    }

    results["models"] = results.get("models", {})
    results["models"]["rf"] = {
        "featureImportance": [{"Feature": _safe(r["Feature"]), "Importance": _safe(r["Importance"])}
                              for _, r in feature_importance.iterrows()],
        "metrics": {
            "accuracy": _safe(round(rf_accuracy, 4)),
            "cvAccuracy": _safe(round(rf_search.best_score_, 4)),
            "cvStd": 0,
            "f1Score": _safe(round(rf_f1, 4)),
            "oobScore": _safe(round(rf_model.oob_score_, 4)) if hasattr(rf_model, 'oob_score_') else 0,
            "precision": _safe(round(precision_score(y_monthly, y_pred_rf, zero_division=0), 4)),
            "recall": _safe(round(recall_score(y_monthly, y_pred_rf, zero_division=0), 4)),
            "nEstimators": _safe(rf_model.n_estimators),
            "rocAuc": _safe(round(rf_auc, 4)),
        },
        "config": _rf_config,
        "thresholds": _rf_thresh,
    }
except Exception as e:
    print(f"RF export error: {e}")

# -- XGBoost --
try:
    _xgb_thresh = {}
    for feat in xgb_thresholds:
        _xgb_thresh[feat] = {
            "value": _safe(round(xgb_thresholds[feat]["threshold"], 3)),
            "lower": _safe(round(xgb_thresholds[feat]["threshold_lower"], 3)),
            "upper": _safe(round(xgb_thresholds[feat]["threshold_upper"], 3)),
            "nSplits": _safe(xgb_thresholds[feat]["n_splits"]),
            "condition": xgb_thresholds[feat].get("direction", "≥"),
            "importance": _safe(round(xgb_thresholds[feat]["importance"], 4)),
            "unit": "m" if "Hm0" in feat else "m/s" if "curr" in feat.lower() or "Wind" in feat else "",
        }

    results["models"] = results.get("models", {})
    results["models"]["xgb"] = {
        "featureImportance": [{"Feature": _safe(r["Feature"]), "Importance": _safe(r["Importance"])}
                              for _, r in xgb_importance.iterrows()],
        "shapValues": [{"Feature": _safe(r["Feature"]), "Mean_SHAP": _safe(r["Mean_SHAP"])}
                       for _, r in shap_importance.iterrows()],
        "metrics": {
            "accuracy": _safe(round(xgb_accuracy, 4)),
            "cvAccuracy": _safe(round(xgb_search.best_score_, 4)),
            "cvStd": 0,
            "f1Score": _safe(round(xgb_f1, 4)),
            "auc": _safe(round(xgb_auc, 4)),
            "precision": _safe(round(precision_score(y_monthly, y_pred_xgb, zero_division=0), 4)),
            "recall": _safe(round(recall_score(y_monthly, y_pred_xgb, zero_division=0), 4)),
            "nEstimators": _safe(xgb_model.n_estimators),
            "maxDepth": _safe(xgb_model.max_depth),
            "learningRate": _safe(xgb_model.learning_rate),
        },
        "thresholds": _xgb_thresh,
    }
except Exception as e:
    print(f"XGB export error: {e}")

# -- HMM --
try:
    # Export ALL feature thresholds with erosion/normal centroids
    _hmm_thresh = {}
    for feat in hmm_thresholds:
        _hmm_thresh[feat] = {
            "value": _safe(round(hmm_thresholds[feat]['threshold'], 3)),
            "direction": hmm_thresholds[feat]['direction'],
            "erosionValue": _safe(round(hmm_thresholds[feat]['erosion_value'], 3)),
            "normalValue": _safe(round(hmm_thresholds[feat]['normal_value'], 3)),
        }

    _state_dist = []
    _sc = env_features_monthly['HMM_State'].value_counts()
    for st in sorted(_sc.index):
        _er = env_features_monthly[env_features_monthly['HMM_State'] == st]['Erosion_Label'].mean()
        _state_dist.append({
            "state": f"State {st}",
            "count": _safe(int(_sc[st])),
            "percentage": _safe(round(_sc[st] / len(env_features_monthly) * 100, 1)),
            "erosionRate": _safe(round(_er * 100, 1)),
        })

    # State centroids (all features, original scale)
    _state_centroids = {}
    try:
        for i in range(n_states):
            _row = state_means_df.loc[f'State_{i}']
            _state_centroids[f"State_{i}"] = {feat: _safe(round(float(_row[feat]), 3)) for feat in model_features if feat in _row.index}
    except Exception:
        pass

    # Component selection data (BIC/AIC for each n_components)
    _component_selection = []
    try:
        for idx, n in enumerate(n_components_range):
            _component_selection.append({
                "nComponents": _safe(int(n)),
                "bic": _safe(round(float(bic_scores[idx]), 1)),
                "aic": _safe(round(float(aic_scores[idx]), 1)),
            })
    except Exception:
        pass

    # Transition matrix export
    _transition_matrix = []
    try:
        for i in range(n_states):
            row = {}
            row["from"] = f"S{i}"
            for j in range(n_states):
                row[f"S{j}"] = _safe(round(float(transition_matrix[i][j]), 4))
            _transition_matrix.append(row)
    except Exception:
        pass

    # Regime stability data
    _regime_stability = {}
    try:
        _regime_stability = {
            "count": _safe(int(len(regime_stability))),
            "mean": _safe(round(float(regime_stability.mean()), 1)),
            "std": _safe(round(float(regime_stability.std()), 1)),
            "min": _safe(int(regime_stability.min())),
            "q25": _safe(int(regime_stability.quantile(0.25))),
            "q50": _safe(int(regime_stability.quantile(0.5))),
            "q75": _safe(int(regime_stability.quantile(0.75))),
            "max": _safe(int(regime_stability.max())),
        }
    except Exception:
        pass

    # Final-state dominance
    _final_state_dominance = []
    try:
        for state_idx in range(n_states):
            pct = float(final_state_dist.get(state_idx, 0))
            _final_state_dominance.append({
                "state": f"State {state_idx}",
                "value": _safe(round(pct, 2)),
            })
    except Exception:
        pass

    # State means for all features
    _all_feats = [f for f in model_features if f in env_features_monthly.columns]
    _normal_means = {}
    _highRisk_means = {}
    for feat in _all_feats:
        try:
            _normal_means[feat] = _safe(round(float(env_features_monthly[env_features_monthly['HMM_State'] != erosion_state][feat].mean()), 3))
            _highRisk_means[feat] = _safe(round(float(env_features_monthly[env_features_monthly['HMM_State'] == erosion_state][feat].mean()), 3))
        except Exception:
            pass

    results["models"] = results.get("models", {})
    results["models"]["hmm"] = {
        "stateDistribution": _state_dist,
        "stateMeans": {
            "normal": _normal_means,
            "highRisk": _highRisk_means,
        },
        "stateCentroids": _state_centroids,
        "componentSelection": _component_selection,
        "transitionMatrix": _transition_matrix,
        "regimeStability": _regime_stability,
        "finalStateDominance": _final_state_dominance,
        "erosionState": _safe(int(erosion_state)),
        "metrics": {
            "nStates": _safe(n_states),
            "accuracy": _safe(round(hmm_accuracy, 4)),
            "logLikelihood": _safe(round(float(hmm_model.score(X_scaled_monthly)), 3)),
            "aic": _safe(round(float(hmm_aic), 3)),
            "bic": _safe(round(float(hmm_bic), 3)),
            "converged": bool(hmm_converged),
            "silhouetteScore": 0,
        },
        "thresholds": _hmm_thresh,
    }
except Exception as e:
    print(f"HMM export error: {e}")

# 12. Thresholds — final consensus table + model comparison + per-variable comparison
try:
    results["thresholds"] = threshold_export
except Exception:
    results["thresholds"] = None

try:
    results["modelComparison"] = model_comparison.to_dict(orient="records")
except Exception:
    results["modelComparison"] = None

try:
    # Build per-variable threshold comparison with numeric values (not string ranges)
    _thresh_comp = []
    _all_drv = sorted(set(hmm_thresholds.keys()) & set(rf_thresholds.keys()) & set(xgb_thresholds.keys()))
    for _drv in _all_drv:
        _h = hmm_thresholds[_drv]
        _r = rf_thresholds[_drv]
        _x = xgb_thresholds[_drv]
        _thresh_comp.append({
            "Variable": _drv,
            "HMM_Threshold": _safe(round(_h["threshold"], 4)),
            "HMM_Range_Lower": _safe(round(_h["threshold_lower"], 4)),
            "HMM_Range_Upper": _safe(round(_h["threshold_upper"], 4)),
            "RF_Threshold": _safe(round(_r["threshold"], 4)),
            "RF_Range_Lower": _safe(round(_r["threshold_lower"], 4)),
            "RF_Range_Upper": _safe(round(_r["threshold_upper"], 4)),
            "XGB_Threshold": _safe(round(_x["threshold"], 4)),
            "XGB_Range_Lower": _safe(round(_x["threshold_lower"], 4)),
            "XGB_Range_Upper": _safe(round(_x["threshold_upper"], 4)),
        })
    results["thresholdComparison"] = _thresh_comp
except Exception:
    results["thresholdComparison"] = None

# 13. Meteorological Threshold Forecasts (SARIMA)
try:
    results["forecasts"] = {
        "metadata": forecast_metadata,
        "variables": {},
    }
    for _fvar in forecast_export:
        _fdata = forecast_export[_fvar]
        _var_export = {
            "threshold": _safe(_fdata["threshold"]),
            "thresholdDirection": _fdata["thresholdDirection"],
            "historicalMean": _safe(_fdata["historicalMean"]),
            "historicalStd": _safe(_fdata["historicalStd"]),
            "historicalMin": _safe(_fdata["historicalMin"]),
            "historicalMax": _safe(_fdata["historicalMax"]),
            "riskLevel": _fdata["riskLevel"],
            "model": _fdata["model"],
            "validation": _fdata.get("validation", {}),
            "horizons": {},
        }
        for _hkey, _hdata in _fdata["horizons"].items():
            _var_export["horizons"][_hkey] = {
                "monthly": _hdata["monthly"],
                "avgForecast": _safe(_hdata["avgForecast"]),
                "peakForecast": _safe(_hdata["peakForecast"]),
                "minForecast": _safe(_hdata["minForecast"]),
                "exceedanceMonths": _safe(_hdata["exceedanceMonths"]),
                "exceedancePct": _safe(_hdata["exceedancePct"]),
                "trendPct": _safe(_hdata["trendPct"]),
            }
        results["forecasts"]["variables"][_fvar] = _var_export
    print(f"✓ Forecast data exported: {len(forecast_export)} variables")
except Exception as e:
    print(f"Forecast export error: {e}")
    results["forecasts"] = None

# ---- Write JSON ----
os.makedirs(os.path.dirname(_RESULTS_PATH), exist_ok=True)
with open(_RESULTS_PATH, "w") as _f:
    json.dump(results, _f, indent=2, default=str)

# Also copy to frontend public dir
os.makedirs(_FRONTEND_PATH, exist_ok=True)
with open(os.path.join(_FRONTEND_PATH, "analysis_results.json"), "w") as _f:
    json.dump(results, _f, indent=2, default=str)

print(f"✓ Results exported to {_RESULTS_PATH}")
print(f"✓ Results copied to {_FRONTEND_PATH}/analysis_results.json")


✓ Forecast data exported: 5 variables
✓ Results exported to D:\Kanjana\Coastal_Research_GitHub\coastalai\backend\results\results_96db02d305d2.json
✓ Results copied to D:\Kanjana\Coastal_Research_GitHub\coastalai\frontend\public\data/analysis_results.json
